# UEMR: chạy A-Z với Athena sạch (`961f1a89`)

**Bản sao của `uemr_final_AZ.ipynb` cho phần B** (`docs/UEMR_B_CHAY_LAI_AZ.md`); bản gốc giữ nguyên để còn tái lập kết quả cũ
(`results/056d388e/`). Khác bản gốc:

- `UNIAV_SOURCE = 'new'`, đầu ra UniAV từ checkpoint `athena_sach/961f1a89/athena_yc2clean.pth` (`ATHENA_CKPT`, `ATHENA_SHA`): chạy
  **`uemr_B1_uniav_athena_sach.ipynb` trước** (runtime riêng). Mục 2 kiểm sha256 của checkpoint và `SOURCE.json` của đầu ra.
- Bảng ghi vào `results/961f1a89/`; cache Omni / partition / fusion theo hash nội dung đầu ra (cơ chế `'new'` có sẵn). Không đụng cache cũ.
- Mục 5: chấm **dev trước**, rồi `assert` các dòng không phụ thuộc UniAV (R0 8.32, R2 27.16, R7 37.52, R1_gt, `uni_M_gt`, `gt_shift`), rồi
  mới in R6. R3 (`uni_M`) dùng K_pred của UniAV nên chỉ in cũ/mới, không assert.
- Mục 8: in kết luận Context Fusion theo tiêu chí ghi trước (`cf_verdict.json`). Mục 13: bảng cũ/mới cho `UEMR_KET_QUA.md` (`bang_cu_moi.csv`).
- `RUN_EXTRA_FUSION`, `RUN_EXP3_FUSION` mặc định **False** (mục 8b: thêm vài giờ A100, ngoài phạm vi phần B).

---

# UEMR: chạy A-Z (lần chạy cuối)

Một notebook từ đầu ra UniAV tới mọi bảng của khoá luận. Tham số lấy từ `configs/final_protocol.json` (giải thích: `docs/UEMR_FINAL_PROTOCOL.md`), không dò lại. Mọi lựa chọn trên dev; val chỉ chấm trong lần chạy cuối.

**Cờ** (ô ngay dưới):

| Cờ | Ý nghĩa |
|---|---|
| `UNIAV_SOURCE` | `'old'`: UniAV hiện tại, dùng layout cache gốc, **regression**: mọi bảng phải trùng `reference_current_uniav` (2 chữ số; fusion ±0.2). `'new'`: đầu ra UniAV mới ở `UNIAV_SRC`, mọi kết quả phụ thuộc UniAV ghi dưới hash của nó |
| `CLEAN_RUN` | Dựng lại từ đầu thay vì dùng kết quả đã lưu: partition, `segments.npz`, `fusion_cache` (với `'old'`: dựng vào thư mục tạm rồi **so** với bản cũ) và encode lại vài clip để so vector. Không xoá store vector đã encode, không train lại lượt fusion đã có `val.json` |
| `RUN_OPTIONAL` | Mục 11 (Exp 4 audio, cần A100) |
| `RUN_EXTRA_FUSION` | Mục 8b: cột (c) cho R1_pred, R2, R4, R5, `uni_M_gt` (1 seed mỗi dòng; encode khoảng 45.000 clip train, A100) |
| `RUN_EXP3_FUSION` | Mục 8b: Exp 3 có Context Fusion, 4 mức nhiễu (1 seed mỗi mức; encode khoảng 31.000 clip train, A100) |
| `ENABLE_GAPFILL` | Giai đoạn 6, chưa duyệt, chưa có code |

**Cách chạy:** **Runtime A100, Run all** (từ trên xuống; mục sau dùng biến của mục trước). Phiên bị ngắt thì chạy lại từ đầu: bước tốn kém tự bỏ qua khi đã có kết quả hợp lệ (encode: store vector; fusion: `val.json`; hash UniAV: `az_state.json`; partition, `fusion_cache`: `run_manifest.json` cùng hash đầu vào). Bảng (CPU, vài phút mỗi mục) luôn tính lại.

**Lần sau chỉ xem lại (không chạy lại):** mọi thứ đã lưu trên Drive (`results/<hash>/`, `fusion_runs_v2/`, `az_state.json`, cache vector và partition). Đặt `CLEAN_RUN = False`, runtime CPU, chạy ô Cờ, 2 ô đầu của mục 0 và ô cuối *In lại mọi bảng*. Run all với `CLEAN_RUN = False` cũng chạy được trên CPU: encode và train fusion tự bỏ qua khi đã có kết quả, bảng tính lại vài phút.

| | `UNIAV_SOURCE = 'old'` | `UNIAV_SOURCE = 'new'` |
|---|---|---|
| GPU dùng cho | mục 8 (train lại 1 lượt D3 để kiểm regression), mục 8b, `CLEAN_RUN` (encode lại vài clip), mục 11 | encode gallery (khoảng 18.000 clip), trainsplit (khoảng 15.000), `pt` (khoảng 7.200), mọi lượt fusion |
| Thời gian ước tính | khoảng 1 giờ; mục 8b lần đầu thêm khoảng 6 giờ (encode khoảng 76.000 clip train), lần sau chỉ đọc lại | khoảng 5–6 giờ (có thể ngắt, chạy lại tiếp) |

Mỗi mục kết thúc bằng `assert` (sai thì dừng) và một dòng trong `results/<hash>/run_manifest.json` (mục, hash đầu vào, hash đầu ra, thời gian). Checklist §14 của `UEMR_FINAL.md` ở cuối notebook.

In [ ]:
# ===== Cờ =====
UNIAV_SOURCE     = 'new'    # 'new': Athena sạch (đầu ra ở UNIAV_SRC) | 'old': chỉ để regression bản cũ
ATHENA_CKPT      = '/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/athena_sach/961f1a89/athena_yc2clean.pth'
ATHENA_SHA       = '961f1a895f4b43c811d3e0cfa31448641989a0a0e39e421ed397682407643d58'
ATHENA_H8        = ATHENA_SHA[:8]
# đầu ra của uemr_B1_uniav_athena_sach.ipynb: gallery 498 video, trainsplit 1.002 video
UNIAV_SRC        = [f'/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/uniav_cache/athena_{ATHENA_H8}',
                    f'/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/uniav_cache/athena_{ATHENA_H8}_train']
CLEAN_RUN        = False    # lần regression đầu: nên True một lần (dựng lại partition, encode lại vài clip để so)
RUN_OPTIONAL     = False    # mục 11: Exp 4 (audio thay bằng im lặng, A100)
RUN_EXTRA_FUSION = False    # mục 8b: cột (c) cho R1_pred, R2, R4, R5, uni_M_gt (1 seed; A100; ngoài phạm vi phần B)
RUN_EXP3_FUSION  = False    # mục 8b: Exp 3 có Context Fusion, 4 mức nhiễu (1 seed; A100; ngoài phạm vi phần B)
ENABLE_GAPFILL   = False    # giai đoạn 6: chưa duyệt
assert UNIAV_SOURCE in ('old', 'new'), UNIAV_SOURCE
assert not ENABLE_GAPFILL, 'uniav_gapfill chưa được duyệt (docs/UEMR_AGENT_CHOT_KET_QUA.md giai đoạn 6): chưa có code'

## 0. Mount, code, môi trường, unit test

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, shutil, json, subprocess, time, glob, zipfile, re, hashlib, platform
from collections import Counter, defaultdict
from pathlib import Path
FUSHION_GIT = 'https://github.com/chien24/Fushion_UEMR.git'
FUSHION     = '/content/code/Fushion_UEMR'

%cd /content
os.makedirs('/content/code', exist_ok=True)
shutil.rmtree(FUSHION, ignore_errors=True)
!git clone -q {FUSHION_GIT} {FUSHION}
COMMIT = subprocess.run(['git', '-C', FUSHION, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print('code:', COMMIT)
if FUSHION not in sys.path:
    sys.path.insert(0, FUSHION)
os.environ['PYTHONPATH'] = FUSHION + ':' + os.environ.get('PYTHONPATH', '')
%cd {FUSHION}
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from omni_retrieval.protocol import P as PROTO, PATH as PROTO_PATH
from omni_retrieval.config import Config
from omni_retrieval.collect import collect, local_npz, rebuilder
from omni_retrieval.evaluate import (coverage_regression, evaluate_partition, gallery_videos, load_index, load_text,
                                     paired_bootstrap, table, video_ranks)
from omni_retrieval.manifest import prepare, read_store, segment_records, write_jsonl, store_keys
from omni_retrieval.partitions import load_partition_file
from omni_retrieval.partitions_uemr import (NOISE_TARGETS, best_tiou, build_noise, coverage, equivalent_tiou, miss_rate,
                                            noise_name, significant_down_to)
from omni_retrieval.uniav_version import fingerprint, hash_of, install, z_overrun
from uemr_fusion.train_cache import (build_train_extra, build_train_partitions, collect_train, gt_by_video, load_train_partitions,
                                     save_train_partitions, train_captions, train_events, train_videos, work_lists)

BASE       = '/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr'
SNAP       = '/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/p6_train_inputs'   # fusion_cache của UniAV cũ
O, U, OT   = f'{BASE}/omni_cache', f'{BASE}/uniav_cache', f'{BASE}/omni_cache_train'
DRIVE_DATA = '/content/drive/MyDrive/Colab Notebooks/code KL/Omni/data/YouCookII'
META, VIDEO_SRC = f'{DRIVE_DATA}/metadata', f'{DRIVE_DATA}/videos'
P0         = f'{O}/common/partitions'          # partition dựng trên GT (gt, gt_noise_*) dùng chung mọi UniAV
SUBSET     = f'{O}/common/subset_videos.json'
DUP_CSV    = f'{O}/ft/analysis/gt_per_query.csv'
STATE_PATH = f'{BASE}/az_state.json'
CKPT_SRC   = '/content/drive/MyDrive/Colab Notebooks/code KL/Omni/checkpoint-1948'
CKPT, ORIG_ADAPTER = '/content/my_checkpoint', '/content/adapters/omniretriever-7b'
BASE_MODEL = '/content/WAVE_HOME/WAVE-7B'
BEATS      = '/content/WAVE_HOME/BEATs_iter3_plus_AS2M_finetuned_on_AS2M_cpt2.pt'
REF        = PROTO['reference_current_uniav']
PROTO_SHA  = hashlib.sha256(open(PROTO_PATH, 'rb').read()).hexdigest()[:16]
OLD        = UNIAV_SOURCE == 'old'

NOISE = [noise_name(t) for t in NOISE_TARGETS]
LABEL = {'global': 'R0 global', 'event_single': 'R1_gt (mean R7)', 'event_single_pred': 'R1_pred (mean R6)',
         'uni_chunk': 'R2 uni_chunk', 'uni_M': 'R3 uni_M', 'rand_M': 'R4 rand_M', 'kmedoids_M': 'R5 kmedoids_M',
         'uniav_pred': 'R6 uniav_pred', 'gt': 'R7 gt (oracle)', 'uni_M_gt': 'uni_M_gt', 'uniav_shift': 'uniav_shift',
         'gt_shift': 'gt_shift', 'uniav_pred+uni_chunk': 'union R6+R2', **{n: n for n in NOISE}}
# file partition nằm ở PG (thư mục theo hash UniAV); uni_chunk, gt_shift không phụ thuộc UniAV nhưng được dựng cùng chỗ
PG_PARTS = ('uniav_pred', 'uni_M', 'rand_M', 'kmedoids_M', 'uni_chunk', 'uniav_shift', 'gt_shift', 'uniav_pred+uni_chunk')
EXP1 = ['global', 'event_single', 'event_single_pred', 'uni_chunk', 'uni_M', 'rand_M', 'kmedoids_M', 'uniav_pred', 'gt',
        'uni_M_gt', 'uniav_shift', 'gt_shift', 'uniav_pred+uni_chunk']
SAME = ('global', 'event_single', 'uni_chunk', 'gt', 'uni_M_gt', 'gt_shift')     # không phụ thuộc UniAV
NEED_FT = ['global', 'gt', 'uni_M_gt', *PG_PARTS, *NOISE]
NEED_PT = ['global', 'gt', 'uni_M', 'uniav_pred']
CHECK = {}                                                       # checklist §14, các mục điền dần

def load_state():
    return json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else {}

def save_state(s):
    json.dump(s, open(STATE_PATH, 'w'), indent=1)

def set_paths(h8):
    """Mọi đường dẫn phụ thuộc UniAV. 'old' giữ layout gốc (cache đã có); 'new' đặt dưới hash h8."""
    global H8, VARIANT, UG, UT, PG, PT, DBH, DBH_PT, OTH, FCH, FC_MAN, RUNSH, RES
    H8 = h8
    if OLD:
        VARIANT, UG, UT, PG, PT = '', f'{U}/ft', f'{U}/ft_train', P0, f'{OT}/common/partitions'
        DBH, DBH_PT, OTH = f'{O}/ft', f'{O}/pt', f'{OT}/ft'
        FCH, FC_MAN, RUNSH = f'{SNAP}/fusion_cache', f'{SNAP}/MANIFEST.json', f'{BASE}/fusion_runs_v2'
    else:
        VARIANT, UG, UT = h8, f'{U}/ft_{h8}', f'{U}/ft_train_{h8}'
        PG, PT = f'{O}/common/partitions_{h8}', f'{OT}/common/partitions_{h8}'
        DBH, DBH_PT, OTH = f'{O}/ft/{h8}', f'{O}/pt/{h8}', f'{OT}/ft/{h8}'
        FCH = f'{BASE}/fusion_cache/ft_{h8}'
        FC_MAN, RUNSH = f'{FCH}/MANIFEST.json', f'{BASE}/fusion_runs_v2/ft_{h8}'
    RES = f'{BASE}/results/{h8 if OLD else ATHENA_H8}'   # Athena sạch: results/<sha256[:8] của checkpoint>
    os.makedirs(RES, exist_ok=True)

H8 = load_state().get(UNIAV_SOURCE, {}).get('hash8')
if H8:
    set_paths(H8)
print('UNIAV_SOURCE', UNIAV_SOURCE, '| hash', H8 or '(chưa có: chạy mục 2)', '| protocol', PROTO_SHA)

def sha256(path, bs=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(bs), b''):
            h.update(chunk)
    return h.hexdigest()

def sh(cmd, check=True):
    """Lệnh shell, in 3.000 ký tự cuối; dừng nếu lỗi."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr)[-3000:]
    if out.strip():
        print(out)
    if check and r.returncode:
        raise RuntimeError(f'lệnh lỗi ({r.returncode}): {cmd[:200]}')
    return r

def omni_cfg(tag, cache_root, **kw):
    return Config.colab(tag=tag, n_videos=500, seed=0, cache_root=cache_root, drive_data=DRIVE_DATA, meta_dir=META,
                        video_src=VIDEO_SRC, wave_path=BASE_MODEL, beats_path=BEATS, adapter_ft=CKPT,
                        adapter_pt=ORIG_ADAPTER, **kw)

def inputs():
    return {'uniav_source': UNIAV_SOURCE, 'uniav_hash8': H8, 'protocol_sha': PROTO_SHA}

PENDING = []                                                      # dòng của mục 1 (trước khi mục 2 biết hash)

def manifest_add(section, outputs, t0, **extra):
    """Một dòng vào run_manifest.json: mục, hash đầu vào, sha256 các file đầu ra (đường dẫn local), thời gian."""
    out = {k: (sha256(v)[:16] if isinstance(v, str) and os.path.isfile(v) else v) for k, v in outputs.items()}
    PENDING.append({'section': section, 'inputs': inputs(), 'outputs': out, 'sec': round(time.time() - t0, 1),
                    'finished': time.strftime('%Y-%m-%d %H:%M:%S'), 'commit': COMMIT, **extra})
    if H8 is None:
        print(f'[manifest] mục {section} xong, ghi sau khi mục 2 biết hash UniAV')
        return
    path = f'{RES}/run_manifest.json'
    m = json.load(open(path)) if os.path.exists(path) else {'uniav_source': UNIAV_SOURCE, 'hash8': H8, 'steps': []}
    for e in PENDING:
        e['inputs']['uniav_hash8'] = H8
    m['steps'] += PENDING
    PENDING.clear()
    json.dump(m, open(path, 'w'), indent=1, default=str)
    print(f'[manifest] mục {section} xong ({time.time() - t0:.0f} s) -> {path}')

def already(section):
    """Mục đã xong với cùng hash đầu vào (và không CLEAN_RUN): bỏ qua phần dựng tốn kém."""
    path = f'{RES}/run_manifest.json'
    if CLEAN_RUN or not os.path.exists(path):
        return False
    return any(s['section'] == section and s['inputs'] == inputs() for s in json.load(open(path))['steps'])

def check_ref(got, ref, what, tol=0.005):
    """Regression: mọi khoá của ref phải có trong got và lệch <= tol."""
    bad = {k: (None if k not in got else round(float(got[k]), 3), v) for k, v in ref.items()
           if k not in got or not abs(float(got[k]) - v) <= tol}
    print(f'[regression] {what}:', f'OK ({len(ref)} số trùng tham chiếu)' if not bad else f'LỆCH (mới, tham chiếu) {bad}')
    assert not bad, f'{what}: khác số tham chiếu {bad}'

def ci(lo, hi):
    return f'[{lo:+.2f}, {hi:+.2f}]'

In [ ]:
# môi trường và unit test (kỳ vọng >= 89 passed)
gpu = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip()
import torch
env = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'torch': torch.__version__,
       'gpu': gpu or 'không có (CPU)', 'commit': COMMIT, 'protocol': f'{PROTO_PATH} ({PROTO_SHA})'}
print(json.dumps(env, indent=1))
r = subprocess.run([sys.executable, '-m', 'pytest', 'omni_retrieval/tests', 'uemr_fusion/tests', '-q', '-p', 'no:cacheprovider'],
                   cwd=FUSHION, capture_output=True, text=True)
print(r.stdout[-1500:])
m = re.search(r'(\d+) passed', r.stdout)
assert r.returncode == 0 and m and int(m.group(1)) >= 89, 'unit test chưa qua: dừng'
ENV = {**env, 'tests passed': int(m.group(1))}

## 1. Protocol: split, video, kẹp GT

`assert`: trainsplit 1.002 video / 7.787 caption, dev 104 / 773, val 394 / 3.030 (`final_protocol.json` `data.splits`); mp4 có đủ; mọi event GT nằm trong [0, độ dài mp4]; gallery val chứa toàn bộ video val và query là toàn bộ caption val (§14).

In [ ]:
t0 = time.time()
sub = json.load(open(SUBSET))['videos']
split_of = {r['video_id']: r['split'] for r in sub}
dur = {r['video_id']: float(r['duration']) for r in sub}
VAL, DEV = gallery_videos(SUBSET, 'val'), gallery_videos(SUBSET, 'dev')
ev_tr = train_events(META)
dur_tr_all = json.load(open(f'{U}/ft_train/durations.json'))          # thời lượng mp4 trainsplit (không phụ thuộc UniAV)
tr_ids = [v for v in train_videos(META) if v in dur_tr_all]
TXT = local_npz(f'{O}/ft', '/content/text_ft', ('text.npz', 'text_dev.npz'))
text_val, text_dev = load_text(TXT), load_text(TXT, 'text_dev.npz')
got = {'trainsplit': (len(tr_ids), len(ev_tr)), 'dev': (len(DEV), len(text_dev['emb'])), 'val': (len(VAL), len(text_val['emb']))}
want = {k: (v['videos'], v['captions']) for k, v in PROTO['data']['splits'].items()}
print('video / caption:', got)
assert got == want, f'split khác protocol: {got} vs {want}'
assert set(text_val['video_id']) == set(VAL), 'query val phải phủ đúng mọi video val'
no_mp4 = [v for v in list(dur) + tr_ids if not os.path.exists(f'{VIDEO_SRC}/{v}.mp4')]
assert not no_mp4, f'{len(no_mp4)} video không có mp4: {no_mp4[:5]}'
gt0 = json.load(open(f'{P0}/gt.json'))
out_of_video = [(v, s) for v in dur for s in gt0[v] if not (0 <= s[0] < s[1] <= dur[v] + 0.01)]
assert not out_of_video, f'GT chưa kẹp theo mp4: {out_of_video[:5]}'
print('GT event (498 video):', sum(len(gt0[v]) for v in dur), '| đều trong [0, duration] | mp4 đủ', len(dur) + len(tr_ids))
CHECK['gallery = mọi video val, query = mọi caption val'] = True
manifest_add('1 protocol', {'subset_videos.json': SUBSET, 'counts': got}, t0)

## 2. UniAV: đầu ra, θ trên dev, mAP, F1, bỏ sót, lệch Z/mp4, thời gian

**Nguồn đầu ra.** Code UniAV trong `Fushion` (`uniav_app`) chạy được inference từ checkpoint Athena (`notebooks/uniav_app_colab.ipynb`, đã chạy 2026-10-11), nhưng cần **runtime riêng** (numpy 2, `transformers` khác môi trường Omni). Vì vậy notebook này **nhận thư mục đầu ra** của UniAV: `segments_pred.jsonl` (có `conf`), `Z/<video>.npz`, `theta.json` và nếu có `raw_proposals.jsonl` (bản code mới ghi thời gian từng video). Với `'old'`: `uniav_cache/ft` và `ft_train`.

`assert`: θ được chọn **trên dev** theo quy tắc (K_pred trung bình gần K_GT nhất); đủ đoạn và Z cho 498 + 1.002 video; mapping giây → index của Z đúng công thức `t = (j + 0.5)·n/256`; K_pred dev xấp xỉ K_GT. `'old'`: F1, AP, K_pred trùng số đã báo.

In [ ]:
t0 = time.time()
state = load_state()
if OLD:
    if CLEAN_RUN or 'old' not in state:
        fp = fingerprint([f'{U}/ft', f'{U}/ft_train'])                  # sha256 của mọi file Z: vài phút
        state['old'] = {'hash8': fp['hash8'], 'fingerprint': fp, 'sources': [f'{U}/ft', f'{U}/ft_train']}
        save_state(state)
    set_paths(state['old']['hash8'])
    THETA_DIRS = [f'{U}/ft']
else:
    assert sha256(ATHENA_CKPT) == ATHENA_SHA, 'checkpoint Athena trên Drive khác ATHENA_SHA'
    for d in UNIAV_SRC:
        s_ = json.load(open(f'{d}/SOURCE.json'))
        assert s_['sha256'] == ATHENA_SHA and s_['run'] == 'yc2clean_E1', f'{d}: không phải đầu ra của Athena {ATHENA_H8}'
        print(d, '| checkpoint', s_['hash8'], '| epoch_seg', s_['epoch_seg'], '| θ', s_['theta']['theta'],
              s_['theta']['chosen_on'], '|', s_['videos'], 'video')
    if (CLEAN_RUN or state.get('new', {}).get('sources') != list(UNIAV_SRC)
            or state['new'].get('athena_sha256') != ATHENA_SHA):
        res = install(UNIAV_SRC, U, {'': sorted(split_of), 'train': tr_ids})
        state['new'] = {'hash8': res['hash8'], 'fingerprint': res['fingerprint'], 'sources': list(UNIAV_SRC),
                        'athena_sha256': ATHENA_SHA,
                        'installed': time.strftime('%Y-%m-%d %H:%M:%S')}
        save_state(state)
    set_paths(state['new']['hash8'])
    shutil.copy2(f'{U}/ft_train/durations.json', f'{UT}/durations.json')
    assert hash_of(UG) == hash_of(UT) == H8
    THETA_DIRS = list(UNIAV_SRC)
print('UniAV', UNIAV_SOURCE, H8, '| gallery', UG, '| train', UT, '| kết quả', RES)
n_z = {d: len(glob.glob(f'{d}/Z/*.npz')) for d in (UG, UT)}
assert n_z == {UG: 498, UT: 1002}, f'thiếu file Z: {n_z}'

# θ phải được chọn trên dev (uniav_app.evaluate.choose_theta ghi theta.json)
th_path = next((f'{d}/theta.json' for d in THETA_DIRS if os.path.exists(f'{d}/theta.json')), None)
assert th_path, ('không có theta.json: bạn UniAV phải gửi kèm theta.json (chạy uniav_app với THETA = None để chọn trên '
                 'dev theo quy tắc của protocol), hoặc chạy lại notebooks/uniav_app_colab.ipynb')
theta_info = json.load(open(th_path))
print('θ:', theta_info)
assert theta_info.get('chosen_on') == 'dev', f'θ không chọn trên dev: {theta_info}'
if OLD:
    assert abs(theta_info['theta'] - REF['theta']) < 1e-9
CHECK['θ chọn trên dev'] = True
# caption bank của checkpoint: code chấm điểm (omni_retrieval, uemr_fusion) không được đọc nó
SKIP = {'uniav_app', 'third_party', 'tests', '_archive', 'uemr2'}
cap_hits = sorted(str(p.relative_to(FUSHION)) for p in Path(FUSHION).rglob('*.py')
                  if not SKIP & set(p.relative_to(FUSHION).parts) and 'caption_pool' in p.read_text(errors='ignore'))
print('code chấm điểm nhắc tới caption_pool của checkpoint:', cap_hits or 'không có')

In [ ]:
from uniav_app.metrics import mean_ap, prf
from uniav_app.detect import timing

gt_all = {**gt0, **gt_by_video(ev_tr)}
ids_by_split = {'trainsplit': tr_ids, 'dev': sorted(DEV), 'val': sorted(VAL)}

def read_pred(*paths):
    out = defaultdict(list)
    for p in paths:
        for line in open(p):
            if line.strip():
                r = json.loads(line)
                out[r['video_id']].append((float(r['ts']), float(r['te']), float(r['conf'])))
    return out

pred = read_pred(f'{UG}/segments_pred.jsonl', f'{UT}/segments_pred.jsonl')
rows2 = []
for split, ids in ids_by_split.items():
    k = np.array([len(pred[v]) for v in ids])
    g = {v: [tuple(s) for s in gt_all[v]] for v in ids}
    p = {v: pred[v] for v in ids}
    f1, ap = prf(p, g, 0.5), mean_ap(p, g, (0.3, 0.5, 0.7))['AP']
    rows2.append({'split': split, 'videos': len(ids), 'K_pred mean': k.mean(), 'K_GT mean': np.mean([len(gt_all[v]) for v in ids]),
                  'K_pred max': int(k.max()), 'K_pred > 16': int((k > 16).sum()), 'F1@0.5': 100 * f1['f1'],
                  'P@0.5': 100 * f1['precision'], 'R@0.5': 100 * f1['recall'],
                  **{f'AP@{t}': 100 * a for t, a in ap.items()},
                  'GT bỏ sót % (tIoU < 0.3)': miss_rate({v: [s[:2] for s in pred[v]] for v in ids}, {v: gt_all[v] for v in ids})})
uni = pd.DataFrame(rows2).set_index('split')
uni.to_csv(f'{RES}/uniav_stats.csv')
display(uni.round(2))
missing = [v for v in list(dur) + tr_ids if not pred[v]]
assert not missing, f'video không có đoạn: {missing[:5]}'
assert abs(uni.loc['dev', 'K_pred mean'] - uni.loc['dev', 'K_GT mean']) < 1.0, 'K_pred dev lệch K_GT quá 1: kiểm θ'
if OLD:
    check_ref({'F1 val': uni.loc['val', 'F1@0.5'], 'F1 dev': uni.loc['dev', 'F1@0.5'], 'F1 train': uni.loc['trainsplit', 'F1@0.5'],
               'K val': uni.loc['val', 'K_pred mean'], 'K dev': uni.loc['dev', 'K_pred mean'], 'AP@0.5 val': uni.loc['val', 'AP@0.5']},
              {'F1 val': 54.4, 'F1 dev': 65.4, 'F1 train': 63.49, 'K val': 7.25, 'K dev': 6.95, 'AP@0.5 val': 36.7},
              'UniAV (NHAN_XET_KET_QUA_UNIAV §2, P6 §2.9)', tol=0.06)

# lệch Z/mp4: t_sec cuối vượt độ dài mp4 quá 1 s
over = z_overrun(f'{UG}/Z', dur) + z_overrun(f'{UT}/Z', {v: dur_tr_all[v] for v in tr_ids})
print('video có t_sec vượt mp4 > 1 s:', pd.DataFrame(over).to_string(index=False) if over else 'không có')
assert not [r for r in over if r['t_sec max'] is None], 'thiếu file Z'
if OLD:
    assert 'wii9jNiNl9Y' in {r['video_id'] for r in over}, 'wii9jNiNl9Y phải có trong danh sách (n 344 so với mp4 333.5 s)'

# thời gian UniAV mỗi video (bản code mới ghi 'sec' trong raw_proposals.jsonl)
raw_p = next((f'{d}/raw_proposals.jsonl' for d in THETA_DIRS if os.path.exists(f'{d}/raw_proposals.jsonl')), None)
TIMING = timing({json.loads(l)['video_id']: json.loads(l) for l in open(raw_p) if l.strip()}) if raw_p else {'videos': 0}
print('thời gian UniAV:', TIMING if TIMING['videos'] else
      'chưa ghi theo từng video (lượt UniAV cũ); tham chiếu log: detector + lưu Z 0.55 s/video, hậu xử lý 0.47 s/video (T4, gồm ghi Drive)')

# mapping giây -> index trên một video val (nhiều event nhất)
v = max(sorted(VAL), key=lambda x: len(gt0[x]))
with np.load(f'{UG}/Z/{v}.npz') as z:
    Z, t, meta = z['z'].astype(np.float32), z['t_sec'], json.loads(str(z['meta']))
assert np.allclose(t, (np.arange(len(t)) + 0.5) * meta['n'] / len(t), atol=1e-3), 'Z không theo t = (j + 0.5)·n/256'
Zn = Z / np.linalg.norm(Z, axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(13, 3))
for ts, te in gt0[v]:
    ax.axvspan(ts, te, color='C2', alpha=0.2)
for ts, te, c in pred[v]:
    ax.axvline(ts, color='C3', lw=1); ax.axvline(te, color='C3', lw=1, ls='--')
ax.plot(t[1:], 1 - (Zn[1:] * Zn[:-1]).sum(1), color='C0', lw=0.8)
ax.set_xlim(0, dur[v]); ax.set_xlabel('giây'); ax.set_ylabel('1 - cos(z_t, z_{t-1})')
ax.set_title(f'{v}: Z {Z.shape}, n {meta["n"]}, mp4 {dur[v]:.1f} s (xanh lá: GT, đỏ: đoạn UniAV)')
fig.savefig(f'{RES}/z_mapping_{v}.png', dpi=120, bbox_inches='tight'); plt.show()
CHECK['mapping giây -> index của Z kiểm trên 1 video'] = True
manifest_add('2 uniav', {'uniav_stats.csv': f'{RES}/uniav_stats.csv', 'theta': theta_info.get('theta'), 'z_overrun': over,
                         'timing': TIMING}, t0)

## 3. Partition

Gallery: R2–R6 (`partitions_uemr`), đối chứng `uniav_shift`, `gt_shift`, union; `gt_noise_*` (Exp 3) dựng trên GT nên dùng chung ở `partitions/`. Trainsplit: R6, R3, R7, R0. `assert` (§2.6 cũ): đủ video, đúng K, đoạn ≥ d_min, trong [0, duration] (code `partitions_uemr.check`); mức nhiễu đạt tIoU ± 0.02; `uni_chunk`, `gt_shift` trùng bản cũ (không phụ thuộc UniAV).

`'old'` + `CLEAN_RUN`: dựng lại mọi thứ vào `/content/clean_parts` và so với file đang dùng.

In [ ]:
t0 = time.time()
def build_gallery(out, force=False):
    sh(f'cd {FUSHION} && python -m omni_retrieval.partitions_uemr --out "{out}" --segments "{UG}/segments_pred.jsonl" '
       f'--z-dir "{UG}/Z" --subset "{SUBSET}" --seed 0' + (' --force' if force else ''))
    shutil.copy2(f'{P0}/gt.json', f'{out}/gt.json')
    sh(f'cd {FUSHION} && python -m omni_retrieval.partitions_uemr --controls --out "{out}" --subset "{SUBSET}" --seed 0')

def build_train(out):
    dur_tr = {v: dur_tr_all[v] for v in tr_ids}
    parts_tr, rows_tr = build_train_partitions(f'{UT}/segments_pred.jsonl', dur_tr, ev_tr)
    save_train_partitions(parts_tr, out, f'{UT}/segments_pred.jsonl', f'{U}/ft_train/durations.json', rows_tr)

def same_parts(a_dir, b_dir, names):
    diff = [n for n in names if load_partition_file(f'{a_dir}/{n}.json', dur) != load_partition_file(f'{b_dir}/{n}.json', dur)]
    assert not diff, f'khác nhau giữa {a_dir} và {b_dir}: {diff}'

if OLD:
    if CLEAN_RUN:
        C = '/content/clean_parts'
        shutil.rmtree(C, ignore_errors=True)
        build_gallery(f'{C}/gallery')
        same_parts(f'{C}/gallery', PG, PG_PARTS)
        sh(f'cd {FUSHION} && python -m omni_retrieval.partitions_uemr --noise --out "{C}/gallery" --subset "{SUBSET}" --seed 0')
        same_parts(f'{C}/gallery', P0, NOISE)
        build_train(f'{C}/train')
        assert all(json.load(open(f'{C}/train/{n}.json')) == json.load(open(f'{PT}/{n}.json'))
                   for n in ('uniav_pred', 'uni_M', 'gt', 'global')), 'partition trainsplit khác'
        print('CLEAN_RUN: partition gallery, gt_noise, trainsplit dựng lại trùng bản đang dùng')
elif CLEAN_RUN or not already('3 partitions'):
    build_gallery(PG, force=CLEAN_RUN)
    same_parts(PG, P0, ('uni_chunk', 'gt_shift'))
    build_train(PT)
src = json.load(open(f'{PG}/SOURCE.json'))
assert src['segments_pred']['sha256'] == sha256(f'{UG}/segments_pred.jsonl'), 'partition không khớp đầu ra UniAV hiện hành'
display(pd.DataFrame([{'partition': n, **{k: v for k, v in r.items() if k not in ('file', 'sha256')}}
                      for n, r in src['partitions'].items()]))
display(pd.DataFrame(src['controls']['summary']))
noise = pd.DataFrame(json.load(open(f'{P0}/SOURCE.json'))['noise']['levels'])
display(noise)
assert (abs(noise['mean tIoU'] - noise['target tIoU']) <= PROTO['exp3_noise']['tol']).all()
if OLD:
    check_ref(dict(zip(noise.partition, noise.sigma)), {'gt_noise_09': 0.05906, 'gt_noise_07': 0.20947,
              'gt_noise_05': 0.43083, 'gt_noise_03': 1.12332}, 'σ của Exp 3', tol=5e-5)
tr = json.load(open(f'{PT}/SOURCE.json'))
display(pd.DataFrame(tr['summary']))
r6 = json.load(open(f'{PG}/uniav_pred.json'))
MISS = {s: miss_rate({v: r6.get(v, []) for v in ids}, {v: gt0[v] for v in ids}) for s, ids in (('val', VAL), ('dev', DEV))}
print('event GT bị R6 bỏ sót (tIoU < 0.3):', {k: round(v, 2) for k, v in MISS.items()})
if OLD:
    check_ref({'miss val': MISS['val']}, {'miss val': 27.8}, 'tỉ lệ bỏ sót val (A4)', tol=0.06)
manifest_add('3 partitions', {'SOURCE.json': f'{PG}/SOURCE.json', 'train SOURCE.json': f'{PT}/SOURCE.json', 'miss %': MISS}, t0)

## 4. Encode Omni: `ft` (val + dev, mọi partition, caption), `pt` (R0, R3, R6, R7)

Ô đầu (CPU) đếm clip còn thiếu. Ô môi trường chỉ cần khi có clip phải encode hoặc `CLEAN_RUN` (A100). `assert`: 0 lỗi, đủ clip; `segments.npz` có đủ partition (đọc qua bản copy local, kiểm cả nội dung). `CLEAN_RUN`: encode lại clip R6, R7 của 3 video val từ đầu và so vector với cache (cos ≥ 0.998).

In [ ]:
t0 = time.time()
cfg_ft = omni_cfg('ft', O, variant=VARIANT, base_partitions=('global', 'gt', 'uni_M_gt'),
                  extra_partitions={**{n: f'{PG}/{n}.json' for n in PG_PARTS}, **{n: f'{P0}/{n}.json' for n in NOISE}})
cfg_pt = omni_cfg('pt', O, variant=VARIANT, base_partitions=('global', 'gt'),
                  extra_partitions={n: f'{PG}/{n}.json' for n in ('uni_M', 'uniav_pred')})
assert str(cfg_ft.out_dir) == DBH and str(cfg_pt.out_dir) == DBH_PT, (cfg_ft.out_dir, cfg_pt.out_dir)
prep_ft = prepare(cfg_ft, write=False)                # write=False: không ghi đè file partition
prep_pt = prepare(cfg_pt, write=False, verbose=False)
assert len(prep_ft['subset']) == 498
TODO = {'ft clip': len(prep_ft['segs_todo']), 'ft caption': len(prep_ft['caps_todo']),
        'pt clip': len(prep_pt['segs_todo']), 'pt caption': len(prep_pt['caps_todo'])}
NEED_GPU = any(TODO.values()) or CLEAN_RUN
print('cần encode:', TODO, '->', 'cần A100: chạy 2 ô dưới' if NEED_GPU else 'không cần GPU, bỏ qua 2 ô dưới')

In [ ]:
# môi trường encode (chỉ khi NEED_GPU); trả về lệnh encode cho TAG 'ft' hoặc 'pt'
from concurrent.futures import ThreadPoolExecutor

def omni_env(tag):
    from google.colab import userdata
    from safetensors import safe_open
    os.environ['HF_TOKEN'] = userdata.get('HF')
    os.environ.pop('HF_XET_HIGH_PERFORMANCE', None)       # chế độ này đệm nhiều GB trong RAM -> tràn RAM
    if not os.path.exists('/content/.fushion_deps_ok'):
        sh(f'pip install -q -r {FUSHION}/requirements.txt')
        open('/content/.fushion_deps_ok', 'w').close()
    sh('pip uninstall -y -q torchao', check=False)
    chk = subprocess.run([sys.executable, '-c', 'import decord, av, peft, torch, transformers; '
                          'print(transformers.__version__, torch.cuda.is_available())'], capture_output=True, text=True)
    last = (chk.stdout.strip().splitlines() or [''])[-1]
    assert chk.returncode == 0 and last.split()[:2] == ['4.51.3', 'True'], f'môi trường encode chưa đúng: {last} {chk.stderr[-500:]}'
    def wave_ok():
        idx = glob.glob(f'{BASE_MODEL}/*.index.json')
        return (os.path.exists(BEATS) and bool(idx) and
                all(os.path.exists(f'{BASE_MODEL}/{s}') for s in set(json.load(open(idx[0]))['weight_map'].values())))
    if not wave_ok():
        sh('hf download nguyenminh04/omni-model --local-dir /content/WAVE_HOME --max-workers 2')
    assert wave_ok(), 'WAVE-7B/BEATs chưa đủ file: chạy lại ô (tải tiếp phần dở)'
    if tag == 'pt':
        if not os.path.exists(f'{ORIG_ADAPTER}/adapter_model.safetensors'):
            sh(f'hf download YunzeLiu/OmniRetriever-7B --local-dir {ORIG_ADAPTER} --quiet')
        adapter = ORIG_ADAPTER
    else:
        if not os.path.exists(f'{CKPT}/adapter_model.safetensors'):
            os.makedirs(CKPT, exist_ok=True)
            for f in os.listdir(CKPT_SRC):
                if os.path.isfile(f'{CKPT_SRC}/{f}'):
                    shutil.copy2(f'{CKPT_SRC}/{f}', f'{CKPT}/{f}')
        adapter = CKPT
    with safe_open(f'{adapter}/adapter_model.safetensors', 'pt') as f:
        keys = list(f.keys())
    heads = [h for h in ('classify_linear', 'beats_ln', 'beats_proj') if any(h in k and 'lora_' not in k for k in keys)]
    assert len(heads) == 3, f'{adapter}: thiếu fusion head {heads}'
    return (f'cd {FUSHION} && python -m omni_retrieval.encode --base-model {BASE_MODEL} --beats-path {BEATS} '
            f'--adapter {adapter} --video-root /content/videos')

def copy_videos(ids):
    os.makedirs('/content/videos', exist_ok=True)
    def one(v):
        src, dst = f'{VIDEO_SRC}/{v}.mp4', f'/content/videos/{v}.mp4'
        if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
            return 0
        shutil.copy2(src, dst + '.part'); os.replace(dst + '.part', dst)
        return 1
    with ThreadPoolExecutor(8) as pool:
        print('đã copy', sum(pool.map(one, list(ids))), 'video về /content/videos')

def encode(modality, manifest, store, log, enc, extra=''):
    """Encode rồi tự đếm phần còn thiếu (| tee che mã lỗi của tiến trình con)."""
    with open(log, 'a') as f:
        f.write(f"\n===== {modality} | {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n")
    get_ipython().system(f'{enc} --modality {modality} --manifest "{manifest}" --store "{store}" {extra} 2>&1 | tee -a "{log}"')
    todo = {json.loads(l)['id'] for l in open(manifest) if l.strip()}
    left = len(todo - store_keys(store, modality))
    print(f'[{modality}] {len(todo)} cần encode | còn thiếu sau encode: {left}')
    if left:
        get_ipython().system(f'tail -n 40 "{log}"')
        raise RuntimeError(f'encode {modality} chưa xong: còn {left}. Xem log {log}, sửa rồi chạy lại ô.')

In [ ]:
# encode phần còn thiếu, gom segments.npz / text.npz (bản local đã kiểm nội dung)
for tag, cfg, prep, out in (('ft', cfg_ft, prep_ft, DBH), ('pt', cfg_pt, prep_pt, DBH_PT)):
    if prep['segs_todo'] or prep['caps_todo']:
        enc = omni_env(tag)
        os.makedirs(out, exist_ok=True)
        if prep['segs_todo']:
            write_jsonl(cfg.segs_todo, prep['segs_todo'])
            copy_videos([r['video_id'] for r in prep['subset']])
            encode('av', cfg.segs_todo, cfg.av_store, f'{out}/az_encode.log', enc, '--num-workers 4')
        if prep['caps_todo']:
            write_jsonl(cfg.caps_todo, prep['caps_todo'])
            encode('text', cfg.caps_todo, cfg.text_store, f'{out}/az_encode.log', enc, '--batch-size 32')
        prep.update(prepare(cfg, write=False, verbose=False))
    if not OLD and (CLEAN_RUN or not zipfile.is_zipfile(f'{out}/segments.npz')):
        print(f'[{tag}] collect ->', collect(cfg, prep))
        if tag == 'ft':
            shutil.copy2(f'{O}/ft/text_dev.npz', f'{out}/text_dev.npz')
        shutil.rmtree(f'/content/db_{tag}_{H8}', ignore_errors=True)
DB = local_npz(DBH, f'/content/db_ft_{H8}', ('segments.npz', 'text.npz', 'text_dev.npz', 'events.npz'), NEED_FT, rebuilder(cfg_ft))
DBP = local_npz(DBH_PT, f'/content/db_pt_{H8}', ('segments.npz', 'text.npz'), NEED_PT, rebuilder(cfg_pt))
with np.load(f'{DB}/segments.npz') as b:
    per_part = dict(Counter(b['partition'].tolist()))
print('segments.npz ft, dòng theo partition:', per_part)
fail = sum(1 for d in (cfg_ft.av_store, cfg_pt.av_store) if os.path.exists(f'{d}/failed.jsonl') for l in open(f'{d}/failed.jsonl') if l.strip())
print('clip lỗi (failed.jsonl):', fail)
assert fail == 0, 'có clip encode lỗi: xem failed.jsonl'

if CLEAN_RUN:                                                   # encode lại từ đầu vài clip, so với cache
    enc = omni_env('ft')
    vids = sorted(VAL)[:3]
    seg = {k: s for k, s in prep_ft['segments'].items() if s['video_id'] in vids
           and any('__uniav_pred__' in r or '__gt__' in r for r in s['refs'])}
    d = '/content/clean_encode'
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    write_jsonl(f'{d}/segs.jsonl', segment_records(seg, set()))
    copy_videos(vids)
    encode('av', f'{d}/segs.jsonl', f'{d}/av', f'{d}/encode.log', enc, '--num-workers 2')
    fresh = read_store(f'{d}/av', 'av')
    with np.load(f'{DB}/segments.npz') as b:
        row = {k: i for i, k in enumerate(b['seg_key'])}
        old = {k: b['emb'][row[k]].astype(np.float32) for k in fresh}
    cos = np.array([fresh[k] @ old[k] / np.linalg.norm(fresh[k]) / np.linalg.norm(old[k]) for k in fresh])
    print(f'CLEAN_RUN encode: {len(cos)} clip, cos min {cos.min():.5f}, mean {cos.mean():.5f}')
    assert cos.min() >= 0.998, 'vector encode lại khác cache'
manifest_add('4 encode', {'segments.npz ft': f'{DB}/segments.npz', 'segments.npz pt': f'{DBP}/segments.npz', 'todo': TODO,
                          'failed': fail}, t0)

## 5. Exp 1 cột (a), (b): bảng, bootstrap theo video, theo độ dài GT, near-duplicate, hồi quy độ phủ, R1

`assert` với `'old'`: mọi R@1 trùng `reference_current_uniav` (val 394, all 498, cột a, dev R6) và các cặp bootstrap chính trùng tổng kết. Với `'new'`: các dòng không phụ thuộc UniAV (R0, R1_gt, R2, R7, `uni_M_gt`, `gt_shift`) trùng tham chiếu.

In [ ]:
t0 = time.time()
text = load_text(DB)
# thứ tự chấm (docs/UEMR_B_CHAY_LAI_AZ.md): dev trước, rồi các dòng không phụ thuộc UniAV, rồi R6
dev_b = {n: evaluate_partition(load_index(DB, n, DEV), text_dev)['R@1'] for n in ('uniav_pred', 'uni_M', 'gt', 'global', 'uni_chunk')}
print('dev R@1 (b):', {LABEL[k]: round(v, 2) for k, v in dev_b.items()})
PAIRS = [('uniav_pred', 'global'), ('uniav_pred', 'event_single_pred'), ('gt', 'event_single'), ('event_single_pred', 'global'),
         ('uniav_pred', 'uni_M'), ('uniav_pred', 'rand_M'), ('uniav_pred', 'kmedoids_M'), ('gt', 'uniav_pred'),
         ('gt', 'uni_M_gt'), ('uni_M', 'uni_chunk'), ('uniav_pred', 'uni_chunk'), ('uniav_pred', 'uniav_shift'),
         ('gt', 'gt_shift'), ('uniav_pred+uni_chunk', 'uni_chunk')]
rows_b, boot_b = table(DB, {n: n for n in EXP1}, [('val394', text, VAL), ('all498', text, None)], PAIRS)
b = pd.DataFrame(rows_b)
b.to_csv(f'{RES}/exp1_b.csv', index=False)
boot = pd.DataFrame(boot_b)
boot.to_csv(f'{RES}/exp1_b_bootstrap.csv', index=False)
rows_a, boot_a = table(DBP, {n: n for n in ('global', 'event_single', 'event_single_pred', 'uni_M', 'uniav_pred', 'gt')},
                       [('val394', load_text(DBP), VAL)], [('uniav_pred', 'event_single_pred'), ('gt', 'event_single')])
a = pd.DataFrame(rows_a)
a.to_csv(f'{RES}/exp1_a_pt.csv', index=False)
r1 = {s: b[b.split == s].set_index('partition')['R@1'].to_dict() for s in ('val394', 'all498')}
ra = a.set_index('partition')['R@1'].to_dict()
if not OLD:
    check_ref({n: r1['val394'][n] for n in SAME}, {n: REF['val394_R@1_b'][n] for n in SAME},
              'dòng không phụ thuộc UniAV (R0, R1_gt, R2, R7, uni_M_gt, gt_shift)')
    print(f"R3 uni_M dùng K_pred của UniAV (protocol uses_uniav), không assert: cũ {REF['val394_R@1_b']['uni_M']:.2f}"
          f" | mới {r1['val394']['uni_M']:.2f}")
COLS = ['vec/video', 'R@1', 'R@5', 'R@10', 'MedR', 'MnR', 'VCMR R@1 (tIoU.5)', 'mom@.5|top1']
for split in ('val394', 'all498'):
    print(f'\n=== Exp 1 cột (b), gallery {split} ===')
    display(b[b.split == split].set_index('partition').rename(index=LABEL)[COLS].round(2))
print('\n=== cột (a) Omni pt, val 394 ==='); display(a.set_index('partition').rename(index=LABEL)[COLS].round(2))
print('\n=== Paired bootstrap R@1, CI 95% theo video ===')
display(boot[['split', 'A', 'B', 'A - B', 'CI (video)', 'p(diff<=0)']].replace(LABEL).round(2))

if OLD:
    check_ref(r1['val394'], REF['val394_R@1_b'], 'Exp 1 (b) val 394')
    check_ref(r1['all498'], REF['all498_R@1_b'], 'Exp 1 (b) all 498')
    check_ref(ra, REF['val394_R@1_a_pt'], 'Exp 1 (a) pt')
    check_ref(dev_b, REF['dev_R@1_b'], 'dev R6')
    bb = boot[boot.split == 'val394'].set_index(['A', 'B'])
    for (x, y), (d, lo, hi) in {('uniav_pred', 'uni_M'): (6.86, 4.97, 8.75), ('uniav_pred', 'uniav_shift'): (7.52, 5.79, 9.28),
                                ('uniav_pred', 'event_single_pred'): (15.64, 13.49, 17.77), ('gt', 'uni_M_gt'): (18.61, 16.49, 20.95),
                                ('uniav_pred+uni_chunk', 'uni_chunk'): (2.61, 1.35, 3.86)}.items():
        check_ref(bb.loc[(x, y)][['A - B', 'ci_low', 'ci_high']].to_dict(), {'A - B': d, 'ci_low': lo, 'ci_high': hi}, f'bootstrap {x} - {y}')
expected = {n: round(float(r1['val394'][n]), 2) for n in ('global', 'uni_M', 'uniav_pred', 'gt')}
json.dump(expected, open(f'{RES}/expected_col_b.json', 'w'), indent=1)

In [ ]:
# theo độ dài GT, theo near-duplicate (val 394), hồi quy độ phủ (RQ2)
from uemr_fusion.evaluate import duplicate_counts
RK = {n: video_ranks(load_index(DB, n, VAL), text)['ranks'] for n in EXP1}
np.savez(f'{RES}/ranks_val394.npz', caption_id=text['caption_id'], video_id=text['video_id'], **RK)
glen = text['gt_te'] - text['gt_ts']
dup = duplicate_counts(text, DUP_CSV)
def by_bins(values, bins, parts=('uni_M', 'uniav_pred', 'gt', 'uni_chunk', 'uni_M_gt')):
    out = []
    for name, lo, hi in bins:
        m = (values >= lo) & (values < hi)
        out.append({'nhóm': name, 'n': int(m.sum()), **{LABEL[p]: 100 * float((RK[p][m] <= 1).mean()) for p in parts}})
    return pd.DataFrame(out)
LEN_BINS = (('<10s', 0, 10), ('10-40s', 10, 40), ('>=40s', 40, np.inf))
DUP_BINS = (('0', 0, 1), ('1-2', 1, 3), ('3-9', 3, 10), ('10-29', 10, 30), ('>=30', 30, np.inf))
by_len, by_dup = by_bins(glen, LEN_BINS), by_bins(np.nan_to_num(dup, nan=-1), DUP_BINS)
by_len.to_csv(f'{RES}/exp1_by_gt_length.csv', index=False); by_dup.to_csv(f'{RES}/exp1_by_near_dup.csv', index=False)
print('=== R@1 theo độ dài GT ==='); display(by_len.round(2))
print('=== R@1 theo số near-duplicate ==='); display(by_dup.round(2))
segs = {n: json.load(open(f'{PG}/{n}.json')) for n in ('uniav_pred', 'uniav_shift', 'uni_M')}
reg = []
for x, y in (('uniav_pred', 'uniav_shift'), ('uniav_pred', 'uni_M')):
    dcov = {v: coverage(segs[x][v], dur[v]) - coverage(segs[y][v], dur[v]) for v in VAL}
    reg.append({'A': LABEL[x], 'B': LABEL[y], **coverage_regression(RK[x], RK[y], text['video_id'], dcov)})
reg = pd.DataFrame(reg)
reg.to_csv(f'{RES}/rq2_coverage_regression.csv', index=False)
print('=== Hồi quy độ phủ ==='); display(reg.round(2))
if OLD:
    l = by_len.set_index('nhóm')
    check_ref({'R6 <10s': l.loc['<10s', LABEL['uniav_pred']], 'R7 <10s': l.loc['<10s', LABEL['gt']],
               'R3 >=40s': l.loc['>=40s', LABEL['uni_M']]}, {'R6 <10s': 21.05, 'R7 <10s': 44.29, 'R3 >=40s': 28.31}, 'theo độ dài GT')
    check_ref({'a': reg.loc[0, 'a (Δcov = 0)']}, {'a': 8.99}, 'hồi quy độ phủ R6 - uniav_shift')
    assert reg.loc[0, 'CI a'] == '[+6.26, +11.88]', reg.loc[0, 'CI a']
manifest_add('5 exp1', {f: f'{RES}/{f}' for f in ('exp1_b.csv', 'exp1_b_bootstrap.csv', 'exp1_a_pt.csv', 'rq2_coverage_regression.csv')}, t0)

## 6. Exp 3 (P5): R@1 theo chất lượng ranh giới

tIoU trung bình (mọi event GT của 498 video: tIoU lớn nhất với đoạn cùng video) của mọi cách chia; R@1 val 394 và all 498; bootstrap từng mức so với `uni_M_gt` (R@1, R@5); theo độ dài GT; đồ thị. Ngưỡng và điểm tương đương tính bằng `partitions_uemr.equivalent_tiou` / `significant_down_to` (tự sắp xếp theo tIoU). `assert`: mức 1.0 là R7; `'old'`: R@1 các mức trùng P5.

In [ ]:
t0 = time.time()
gt498 = {v: gt0[v] for v in dur}
def part_file(n):
    return f'{P0}/{n}.json' if (n in NOISE or n in ('gt', 'uni_M_gt', 'global')) else f'{PG}/{n}.json'
P5_PARTS = ['gt', *NOISE, 'uni_M_gt', 'uniav_pred', 'uni_M', 'uni_chunk', 'global', 'rand_M', 'kmedoids_M']
tiou = {n: float(best_tiou(json.load(open(part_file(n))), gt498).mean()) for n in P5_PARTS if os.path.exists(part_file(n))}
assert abs(tiou['gt'] - 1.0) < 1e-9, 'mức 1.0 phải là R7'
rows6, _ = table(DB, {n: n for n in P5_PARTS}, [('val394', text, VAL), ('all498', text, None)])
p5 = pd.DataFrame(rows6)
p5['mean tIoU'] = p5.partition.map(tiou)
p5.to_csv(f'{RES}/p5_table.csv', index=False)
display(p5[p5.split == 'val394'].set_index('partition')[['mean tIoU', 'vec/video', 'R@1', 'R@5', 'R@10', 'VCMR R@1 (tIoU.5)', 'mom@.5|top1']].round(3))
base = RK['uni_M_gt']
lev = []
for n in ['gt', *NOISE]:
    rk = RK[n] if n in RK else video_ranks(load_index(DB, n, VAL), text)['ranks']
    b1, b5 = (paired_bootstrap(rk, base, k, 1000, 0, 'video', text['video_id']) for k in (1, 5))
    lev.append({'mức': n, 'tIoU': tiou[n], 'R@1': 100 * float((rk <= 1).mean()), 'Δ R@1': b1['diff'], 'CI R@1': ci(b1['ci_low'], b1['ci_high']),
                'ci_low R@1': b1['ci_low'], 'Δ R@5': b5['diff'], 'CI R@5': ci(b5['ci_low'], b5['ci_high'])})
    RK[n] = rk
lev = pd.DataFrame(lev)
lev.to_csv(f'{RES}/p5_bootstrap_vs_uni_M_gt.csv', index=False)
display(lev.drop(columns='ci_low R@1').round(3))
val5 = p5[p5.split == 'val394'].set_index('partition')['R@1']
curve = [(tiou[n], val5[n]) for n in ['gt', *NOISE]]
TARGET = {'gt': 1.0, **dict(zip(NOISE, NOISE_TARGETS))}          # mức theo tIoU mục tiêu (đạt được lệch < 0.02)
summary = {'significant down to tIoU': significant_down_to(list(zip(lev['mức'].map(TARGET), lev['ci_low R@1']))),
           'R6 equivalent tIoU': equivalent_tiou(curve, val5['uniav_pred']),
           'break-even with uni_M_gt (tIoU)': equivalent_tiou(curve, val5['uni_M_gt']),
           'R6 below curve (points)': float(np.interp(tiou['uniav_pred'], *zip(*sorted(curve)))) - val5['uniav_pred']}
json.dump(summary, open(f'{RES}/p5_summary.json', 'w'), indent=1)
print(json.dumps({k: round(v, 3) if isinstance(v, float) else v for k, v in summary.items()}, indent=1))
byl = by_bins(glen, LEN_BINS, parts=('gt', *NOISE, 'uni_M_gt', 'uniav_pred', 'uni_M', 'uni_chunk'))
byl.to_csv(f'{RES}/p5_by_gt_length.csv', index=False); display(byl.round(2))
fig, ax = plt.subplots(figsize=(7, 4.5))
xs, ys = zip(*sorted(curve))
ax.plot(xs, ys, 'o-', color='C0', label='GT nhiễu (K_GT)')
for n, mk, c in (('uni_M_gt', 's', 'C1'), ('uniav_pred', 'D', 'C3'), ('uni_M', 's', 'C2'), ('uni_chunk', '^', 'C4')):
    ax.plot(tiou[n], val5[n], mk, color=c, ms=8, label=LABEL[n])
ax.axhline(val5['global'], ls=':', color='gray', label='R0 global')
ax.set_xlabel('tIoU trung bình với GT'); ax.set_ylabel('R@1 (%), gallery val 394'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.savefig(f'{RES}/p5_r1_vs_tiou.png', dpi=150, bbox_inches='tight'); plt.show()
if OLD:
    check_ref({n: val5[n] for n in NOISE}, REF['p5_val394_R@1'], 'Exp 3 (P5)')
    assert summary['significant down to tIoU'] == 0.5
    check_ref({'eq': summary['R6 equivalent tIoU'], 'be': summary['break-even with uni_M_gt (tIoU)']}, {'eq': 0.41, 'be': 0.26},
              'P5 nội suy', tol=0.006)
manifest_add('6 exp3', {f: f'{RES}/{f}' for f in ('p5_table.csv', 'p5_bootstrap_vs_uni_M_gt.csv', 'p5_summary.json')}, t0)

## 7. Trainsplit cho fusion: encode, `fusion_cache`

`'old'`: dùng snapshot `p6_train_inputs/` (kiểm sha256 từng file với `MANIFEST.json`). `'new'`: encode R6, R3 mới trên 1.002 video trainsplit (`gt`, `global` dùng lại vector cũ; A100), dựng `fusion_cache/ft_<hash>` ở local, parity với cột (b) của mục 5, chép lên Drive kèm `MANIFEST.json`. `assert`: parity, sha256, 1.002/1.002 video train có vector global.

In [ ]:
t0 = time.time()
if not OLD and (CLEAN_RUN or not os.path.exists(FC_MAN)):
    cfg_tr = omni_cfg('ft', OT, variant=VARIANT)
    assert str(cfg_tr.out_dir) == OTH
    parts_tr = load_train_partitions(PT, ('uniav_pred', 'uni_M', 'gt', 'global'))
    dur_tr = {v: dur_tr_all[v] for v in parts_tr['uniav_pred']}
    caps_tr = train_captions(ev_tr, dur_tr)
    wl = work_lists(parts_tr, caps_tr, cfg_tr.av_store, cfg_tr.text_store, cfg_tr.segs_todo, cfg_tr.caps_todo)
    print('trainsplit: video', len(dur_tr), '| caption', len(caps_tr), '| clip mới', wl['segs_todo'], '| caption mới', wl['caps_todo'])
    os.makedirs(OTH, exist_ok=True)
    if wl['segs_todo'] or wl['caps_todo']:
        enc = omni_env('ft')
        if wl['segs_todo']:
            copy_videos(dur_tr)
            encode('av', cfg_tr.segs_todo, cfg_tr.av_store, f'{OTH}/az_encode.log', enc, '--num-workers 4')
        if wl['caps_todo']:
            encode('text', cfg_tr.caps_todo, cfg_tr.text_store, f'{OTH}/az_encode.log', enc, '--batch-size 32')
    print('[train] collect ->', collect_train(cfg_tr, parts_tr, caps_tr, dur_tr))
    OTL = local_npz(OTH, f'/content/db_train_{H8}', ('segments.npz', 'text.npz'), ('uniav_pred', 'uni_M', 'gt', 'global'))
    FCL = f'/content/fusion_cache_{H8}'
    shutil.rmtree(FCL, ignore_errors=True)
    sh(f'cd {FUSHION} && python -m uemr_fusion.build_cache --out "{FCL}" --gallery-omni "{DB}" --subset "{SUBSET}" '
       f'--gallery-z "{UG}/Z" --train-omni "{OTL}" --train-z "{UT}/Z" --train-durations "{UT}/durations.json" '
       f"--expected '{json.dumps(expected)}'")
    src = json.load(open(f'{FCL}/SOURCE.json'))
    assert src.get('parity'), 'parity fusion_cache lỗi: xem log'
    os.makedirs(FCH, exist_ok=True)
    man = {'created': time.strftime('%Y-%m-%d %H:%M:%S'), 'uniav_hash8': H8, 'parity': src['parity'], 'files': {}}
    for f in sorted(os.listdir(FCL)):
        man['files'][f] = {'bytes': os.path.getsize(f'{FCL}/{f}'), 'sha256': sha256(f'{FCL}/{f}')}
        if os.path.exists(f'{FCH}/{f}'):
            os.remove(f'{FCH}/{f}')
        shutil.copy2(f'{FCL}/{f}', f'{FCH}/{f}')          # ghi từ bản local, không os.replace trên Drive
    json.dump(man, open(FC_MAN, 'w'), indent=1)

def fetch_fusion_cache():
    """fusion_cache về local, sha256 từng file khớp MANIFEST.json (copy lại một lần nếu Drive đọc hụt)."""
    man = json.load(open(FC_MAN))
    root, local = os.path.dirname(FC_MAN), f'/content/fusion_cache_{H8}'
    os.makedirs(local, exist_ok=True)
    if OLD:   # snapshot P6: khoá 'fusion_cache/<file>' và phải cùng thế hệ với fusion_cache/SOURCE.json
        assert man['files']['fusion_cache/SOURCE.json']['sha256'] == sha256(f'{BASE}/fusion_cache/SOURCE.json'), \
            'snapshot p6_train_inputs cũ hơn fusion_cache'
    for rel, m in man['files'].items():
        if OLD and not rel.startswith('fusion_cache/'):
            continue
        dst = f'{local}/{os.path.basename(rel)}'
        for _ in range(2):
            if os.path.exists(dst) and os.path.getsize(dst) == m['bytes'] and sha256(dst) == m['sha256']:
                break
            shutil.copy2(f'{root}/{rel}', dst)
        assert sha256(dst) == m['sha256'], f'{root}/{rel} hỏng trên Drive'
    return local, man

FCL, man = fetch_fusion_cache()
import torch
eg_ok = torch.load(f'{FCL}/train_uniav_pred.pt', weights_only=False)['eg_ok']
assert eg_ok.all() and len(eg_ok) == 1002, 'train thiếu vector global (c_g)'
print('fusion_cache', FCL, '| parity:', [(r['partition'], r.get('expected R@1'), r['R@1 (E)']) for r in man['parity']])
manifest_add('7 fusion_cache', {'MANIFEST.json': FC_MAN, 'files': len(man['files'])}, t0)

## 8. Context Fusion (D3), Exp 1 cột (c), Exp 2, đối chứng (A100 hoặc L4)

Cấu hình đã chốt (`final_protocol.json` `fusion`; mặc định của `train.py` trùng nó, có test). Lượt có `val.json` thì bỏ qua (`'old'`: mọi lượt của P6 đã có). `'old'`: train lại **một** lượt D3 seed 0 vào `results/<hash>/fusion_regress/` để kiểm regression (≤ 0.2 điểm). `assert`: overfit > 90 %; cột (b) của mỗi lượt = mục 5.

Epoch chọn theo dev R@1 (hoà lấy epoch sớm hơn). Cột (c) cho R1_pred, R2, R4, R5, `uni_M_gt` và Exp 3 có Context Fusion: mục 8b.

In [ ]:
t0 = time.time()
from uemr_fusion.evaluate import seed_mean_bootstrap
shutil.copy2(DUP_CSV, '/content/gt_per_query.csv')
os.makedirs(RUNSH, exist_ok=True)
F = PROTO['fusion']

def train(name, partition, flags=(), runs=None, cache=None):
    run = f'{runs or RUNSH}/{name}'
    out = f'{run}/overfit.json' if any(f.startswith('--overfit') for f in flags) else f'{run}/val.json'
    if CLEAN_RUN or not os.path.exists(out):     # đã có kết quả trên Drive thì chỉ đọc lại, không cần GPU
        assert torch.cuda.is_available(), 'không có GPU: Runtime -> A100 hoặc L4'
        os.makedirs(run, exist_ok=True)
        args = [f'--cache "{cache or FCL}"', f'--out "{run}"', '--dup-csv /content/gt_per_query.csv', f'--partition {partition}', *flags]
        get_ipython().system(f'cd {FUSHION} && python -m uemr_fusion.train {" ".join(args)} 2>&1 | tee -a "{run}/train.log"')
    if not os.path.exists(out):
        raise RuntimeError(f'{name}: không có {os.path.basename(out)}, xem {run}/train.log')
    return json.load(open(out))

def load(name, runs=None):
    p = f'{runs or RUNSH}/{name}/val.json'
    return json.load(open(p)) if os.path.exists(p) else None

of = train('overfit_R6', 'uniav_pred', ['--overfit 32', '--steps 200', '--tau-s 0.02', '--lam 0.5', '--no-global',
                                         f"--alpha-lr {F['overfit_check']['alpha_lr']}"])
print('overfit R@1:', of['R@1 after'])
assert of['pass (> 90)'], 'overfit chưa đạt 90 %: dừng, gửi log'
seeds = F['seeds']
RUNS = ([('D3' if s == seeds[0] else f'D3_seed{s}', 'uniav_pred', [f'--seed {s}']) for s in seeds]
        + [('E_uni_M' if s == seeds[0] else f'E_uni_M_seed{s}', 'uni_M', [f'--seed {s}']) for s in seeds]
        + [('E_global' if s == seeds[0] else f'E_global_seed{s}', 'global', [f'--seed {s}']) for s in seeds]
        + [('E_gt', 'gt', [])]
        + [(n, 'uniav_pred', fl) for n, fl in F['exp2_runs'].items() if n != 'D3']
        + [(f'E_{p}_lvid', p, F['lvid_only_runs']['flags']) for p in F['lvid_only_runs']['partitions']])
for name, part, flags in RUNS:
    r = train(name, part, flags)
    assert abs(r['val394 (b)']['R@1'] - expected.get(part, r['val394 (b)']['R@1'])) < 0.006, f"{name}: cột (b) khác mục 5"
if OLD:   # regression: train lại D3 seed 0 từ đầu, so với lượt P6
    rr = train('D3', 'uniav_pred', ['--seed 0'], runs=f'{RES}/fusion_regress')
    check_ref({'D3 (c)': rr['val394 (c)']['R@1'], 'D3 (b)': rr['val394 (b)']['R@1']},
              {'D3 (c)': REF['fusion_D3_val394_R@1_seeds'][0], 'D3 (b)': REF['val394_R@1_b']['uniav_pred']},
              'fusion D3 seed 0 train lại', tol=REF['fusion_D3_seed_tol'])

In [ ]:
# bảng cột (c), Exp 2, đối chứng, chỉ-L_vid
rows_c = []
for base_name, part in (('D3', 'uniav_pred'), ('E_uni_M', 'uni_M'), ('E_global', 'global'), ('E_gt', 'gt')):
    names = [base_name] + [f'{base_name}_seed{s}' for s in seeds[1:]]
    have = [n for n in names if load(n)]
    c = np.array([load(n)['val394 (c)']['R@1'] for n in have])
    row = {'cách chia': LABEL[part], 'lượt': base_name, 'seed': len(have), '(b)': load(have[0])['val394 (b)']['R@1'],
           '(c) mean': c.mean(), '(c) std': c.std(ddof=1) if len(c) > 1 else np.nan, 'alpha': ', '.join(f"{load(n)['alpha']:+.2f}" for n in have),
           'VCMR (c)': np.mean([load(n)['val394 (c)']['VCMR R@1 (tIoU.5)'] for n in have])}
    row['(c) - (b)'] = row['(c) mean'] - row['(b)']
    if len(have) > 1:
        m = [r for r in seed_mean_bootstrap([f'{RUNSH}/{n}/ranks.npz' for n in have])
             if r['gallery'] == 'val394' and r['run'].startswith('mean')][0]
        row['CI mean seed (video)'] = ci(m['ci_low'], m['ci_high'])
        row['ci_low'] = m['ci_low']
    else:
        bs = load(have[0])['val394 bootstrap (c)-(b) by video']
        row['CI mean seed (video)'] = ci(bs['ci_low'], bs['ci_high'])
        row['ci_low'] = bs['ci_low']
    rows_c.append(row)
exp1_c = pd.DataFrame(rows_c)
exp1_c.to_csv(f'{RES}/exp1_c.csv', index=False)
print('=== Exp 1 cột (c), val 394 ==='); display(exp1_c.round(2))
d3c = exp1_c.set_index('lượt').loc['D3']
CF_VERDICT = {'quy tắc (ghi trước khi chấm val)': 'R6 + CF (trung bình 3 seed D3) hơn R6, cận dưới CI 95% > 0 '
                                                '(bootstrap theo video, 1.000 lần, seed 0, val 394)',
              'seed': int(d3c['seed']), 'R6 (b)': round(float(d3c['(b)']), 2), 'R6 + CF (c) mean': round(float(d3c['(c) mean']), 2),
              'Δ': round(float(d3c['(c) - (b)']), 2), 'CI': d3c['CI mean seed (video)'],
              'Context Fusion vào phương pháp chính': bool(d3c['seed'] == len(seeds) and d3c['ci_low'] > 0)}
CF_VERDICT['phương pháp chính'] = 'R6 + Context Fusion (D3)' if CF_VERDICT['Context Fusion vào phương pháp chính'] else 'R6 MaxSim'
json.dump(CF_VERDICT, open(f'{RES}/cf_verdict.json', 'w'), indent=1, ensure_ascii=False)
print(json.dumps(CF_VERDICT, indent=1, ensure_ascii=False))
E2 = [('Omni thuần', None), ('+ bước 1 (cross)', 'F_cross_only'), ('+ bước 2 (self)', 'D1_ts0.05'), ('+ L_evt', 'D2'),
      ('+ c_g (đầy đủ)', 'D3'), ('Z xáo trộn', 'F_shuffle_z'), ('δ = 0', 'F_delta0')]
d3 = load('D3')
exp2 = []
for label, n in E2:
    r = d3 if n is None else load(n)
    col = 'val394 (b)' if n is None else 'val394 (c)'
    bs = None if n is None else r['val394 bootstrap (c)-(b) by video']
    dev = load('D1_ts0.05')['dev_without_fusion']['R@1'] if n is None else r['dev_best']['R@1']   # cột (b) không c_g: 49.68
    exp2.append({'setting': label, 'lượt': n or 'cột (b)', 'dev R@1': dev,
                 'R@1': r[col]['R@1'], 'R@5': r[col]['R@5'], 'R@10': r[col]['R@10'],
                 '(c) - (b)': None if bs is None else bs['diff'], 'CI (video)': None if bs is None else ci(bs['ci_low'], bs['ci_high']),
                 'alpha': 0.0 if n is None else r['alpha']})
exp2 = pd.DataFrame(exp2)
exp2.to_csv(f'{RES}/exp2.csv', index=False)
print('=== Exp 2 (R6, val 394) ==='); display(exp2.round(2))
pairs = []
for other in ('F_shuffle_z', 'F_cross_only', 'D1_ts0.05', 'D2', 'F_delta0'):
    x, y = np.load(f'{RUNSH}/D3/ranks.npz'), np.load(f'{RUNSH}/{other}/ranks.npz')
    bs = paired_bootstrap(x['ranks_c_val394'], y['ranks_c_val394'], 1, 1000, 0, 'video', x['video_id'])
    pairs.append({'A': 'D3', 'B': other, 'A - B': bs['diff'], 'CI (video)': ci(bs['ci_low'], bs['ci_high'])})
pairs = pd.DataFrame(pairs)
pairs.to_csv(f'{RES}/exp2_pairs.csv', index=False); display(pairs.round(2))
lv = pd.DataFrame([{'lượt': f'E_{p}_lvid', 'cách chia': LABEL[p], '(b)': load(f'E_{p}_lvid')['val394 (b)']['R@1'],
                    '(c)': load(f'E_{p}_lvid')['val394 (c)']['R@1'],
                    'CI (video)': ci(*[load(f'E_{p}_lvid')['val394 bootstrap (c)-(b) by video'][k] for k in ('ci_low', 'ci_high')])}
                   for p in F['lvid_only_runs']['partitions']])
lv.to_csv(f'{RES}/lvid_only.csv', index=False); display(lv.round(2))
order = exp1_c.set_index('lượt')['(c) mean']
print('thứ tự R0 < R3 < R6 < R7 sau fusion:', 'giữ' if order['E_global'] < order['E_uni_M'] < order['D3'] < order['E_gt'] else 'KHÔNG giữ (ghi vào nhận xét)')
if OLD:
    check_ref({f'D3 seed {s}': load('D3' if s == seeds[0] else f'D3_seed{s}')['val394 (c)']['R@1'] for s in seeds},
              dict(zip([f'D3 seed {s}' for s in seeds], REF['fusion_D3_val394_R@1_seeds'])), 'fusion D3 3 seed (lượt P6)')
CHECK['batch một caption mỗi video'] = True                      # one_per_video_batches, test_data.py
manifest_add('8 fusion', {f: f'{RES}/{f}' for f in ('exp1_c.csv', 'exp2.csv', 'exp2_pairs.csv', 'lvid_only.csv')}, t0,
             runs=RUNSH)

## 8b. Cột (c) đầy đủ và Exp 3 có Context Fusion (`RUN_EXTRA_FUSION`, `RUN_EXP3_FUSION`; A100)

- **Cột (c)** cho R1_pred, R2, R4, R5, `uni_M_gt` và **đường có Context Fusion của Exp 3** (4 mức nhiễu), cùng cấu hình D3 như mục 8, **1 seed** (seed 0) mỗi dòng như R7 (`UEMR_FINAL.md` §10: 3 seed chỉ cho R0, R3, R6). Mức 1.0 của Exp 3 là lượt `E_gt` của mục 8.
- **Phía train dùng đúng hàm của gallery:** R2, R4, R5 bằng `partitions_uemr.build` (cùng seed theo video, K-Medoids trên Z train), `uni_M_gt` như `build_base`, các mức nhiễu bằng `build_noise` trên `gt.json` của trainsplit (σ dò lại trên train để đạt cùng tIoU đích ±0.02; in kèm σ của gallery). R1_pred là trung bình vector R6 (không encode thêm).
- **Thư mục riêng, không đụng cache đang có:** partition `<PT>/extra/`, vector `omni_cache_train/ft/<variant>_extra/` (kho `raw/` dùng chung, clip đã encode được dùng lại), `fusion_cache_extra/<hash>/`, lượt `<RUNSH>/extra/`.
- **Chi phí lần đầu:** encode khoảng 45.000 clip (cột c) + 31.000 (Exp 3), khoảng 6 giờ A100; ngắt thì chạy lại (store vector và `val.json` tự bỏ qua phần đã xong). Với UniAV mới chỉ R4, R5 (khoảng 15.000 clip) phụ thuộc UniAV phải encode lại; R2, `uni_M_gt` và các mức nhiễu dùng lại.
- **Quy tắc đọc Exp 3, ghi trước khi chạy:** Context Fusion "làm hệ thống bền hơn với ranh giới kém" nếu ở mức tIoU 0.5 hoặc 0.3, (c) − (b) **lớn hơn** ở mức 1.0 **và** khoảng tin cậy của (c) − (b) ở mức đó **không chứa 0**. Ngược lại: không có bằng chứng.
- `assert`: mọi partition train có đủ 1.002 video và qua `check`; σ train đạt tIoU đích; parity cột (b) trên `fusion_cache_extra` (R1_pred ±0.1 vì vector trung bình lưu fp16); 1.002 video train có `c_g`; cột (b) của mỗi lượt = mục 5 / mục 6.

In [ ]:
# 8b-1: partition trainsplit còn thiếu, encode, fusion_cache_extra
t0 = time.time()
EXTRA_C = ['event_single_pred', 'uni_chunk', 'rand_M', 'kmedoids_M', 'uni_M_gt'] if RUN_EXTRA_FUSION else []
EXTRA_N = list(NOISE) if RUN_EXP3_FUSION else []
EXTRA = EXTRA_C + EXTRA_N

def publish(local, dst, man_path, **extra):
    """Chép thư mục local lên Drive kèm MANIFEST.json (sha256 từng file); ghi từ bản local, không os.replace trên Drive."""
    os.makedirs(dst, exist_ok=True)
    man = {'created': time.strftime('%Y-%m-%d %H:%M:%S'), 'uniav_hash8': H8, **extra, 'files': {}}
    for f in sorted(os.listdir(local)):
        man['files'][f] = {'bytes': os.path.getsize(f'{local}/{f}'), 'sha256': sha256(f'{local}/{f}')}
        if os.path.exists(f'{dst}/{f}'):
            os.remove(f'{dst}/{f}')
        shutil.copy2(f'{local}/{f}', f'{dst}/{f}')
    json.dump(man, open(man_path, 'w'), indent=1, default=str)

def fetch(man_path, local):
    """Về local, sha256 từng file khớp MANIFEST.json (copy lại một lần nếu Drive đọc hụt)."""
    man, root = json.load(open(man_path)), os.path.dirname(man_path)
    os.makedirs(local, exist_ok=True)
    for f, m in man['files'].items():
        dst = f'{local}/{f}'
        for _ in range(2):
            if os.path.exists(dst) and os.path.getsize(dst) == m['bytes'] and sha256(dst) == m['sha256']:
                break
            shutil.copy2(f'{root}/{f}', dst)
        assert sha256(dst) == m['sha256'], f'{root}/{f} hỏng trên Drive'
    return local

if EXTRA:
    PTX = f'{PT}/extra'
    cfg_x = omni_cfg('ft', OT, variant=f'{VARIANT}_extra' if VARIANT else 'extra')
    OTX = str(cfg_x.out_dir)
    FCX = f'{BASE}/fusion_cache_extra/{H8}'
    FCX_MAN, RUNSX = f'{FCX}/MANIFEST.json', f'{RUNSH}/extra'
    expected_x = {**{n: round(float(r1['val394'][n]), 2) for n in EXTRA_C}, **{n: round(float(val5[n]), 2) for n in EXTRA_N}}
    have = json.load(open(FCX_MAN)).get('partitions', []) if os.path.exists(FCX_MAN) else []
    if CLEAN_RUN or not set(EXTRA) <= set(have):
        dur_tr = {v: dur_tr_all[v] for v in tr_ids}
        base_tr = load_train_partitions(PT, ('uniav_pred', 'global'))       # nguồn của R1_pred, c_g: đã encode
        parts_x, rows_x, km_x = build_train_extra(f'{UT}/segments_pred.jsonl', dur_tr, ev_tr, f'{UT}/Z')
        display(pd.DataFrame(rows_x))
        noise_x, lev_x = {}, []
        if EXTRA_N:
            noise_x, lev_x, _ = build_noise(PT, dur_tr)                       # gt.json của trainsplit, seed 0 như gallery
            gal_sigma = {r['partition']: r['sigma'] for r in json.load(open(f'{P0}/SOURCE.json'))['noise']['levels']}
            display(pd.DataFrame(lev_x).assign(**{'sigma gallery': lambda d: d.partition.map(gal_sigma)}))
            assert all(abs(r['mean tIoU'] - r['target tIoU']) <= PROTO['exp3_noise']['tol'] for r in lev_x), 'σ train lệch tIoU đích'
        save_train_partitions({**parts_x, **noise_x}, PTX, f'{UT}/segments_pred.jsonl', f'{UT}/durations.json', rows_x + lev_x)
        parts_enc = {**base_tr, **{n: parts_x[n] for n in EXTRA_C if n in parts_x}, **noise_x}
        caps_tr = train_captions(ev_tr, dur_tr)
        wl = work_lists(parts_enc, caps_tr, cfg_x.av_store, cfg_x.text_store, cfg_x.segs_todo, cfg_x.caps_todo)
        print('trainsplit extra: video', len(dur_tr), '| clip mới', wl['segs_todo'], f"(khoảng {wl['segs_todo'] * 0.26 / 3600:.1f} giờ A100)",
              '| caption mới', wl['caps_todo'])
        os.makedirs(OTX, exist_ok=True)
        if wl['segs_todo'] or wl['caps_todo']:
            enc = omni_env('ft')
            if wl['segs_todo']:
                copy_videos(dur_tr)
                encode('av', cfg_x.segs_todo, cfg_x.av_store, f'{OTX}/az_encode.log', enc, '--num-workers 4')
            if wl['caps_todo']:
                encode('text', cfg_x.caps_todo, cfg_x.text_store, f'{OTX}/az_encode.log', enc, '--batch-size 32')
        print('[train extra] collect ->', collect_train(cfg_x, parts_enc, caps_tr, dur_tr))
        shutil.rmtree(f'/content/db_train_x_{H8}', ignore_errors=True)
        OTXL = local_npz(OTX, f'/content/db_train_x_{H8}', ('segments.npz', 'text.npz'), tuple(parts_enc))
        FXL = f'/content/fusion_cache_x_{H8}'
        shutil.rmtree(FXL, ignore_errors=True)
        sh(f'cd {FUSHION} && python -m uemr_fusion.build_cache --out "{FXL}" --gallery-omni "{DB}" --subset "{SUBSET}" '
           f'--gallery-z "{UG}/Z" --train-omni "{OTXL}" --train-z "{UT}/Z" --train-durations "{UT}/durations.json" '
           f"--partitions {' '.join(EXTRA)} --expected '{json.dumps(expected_x)}'")
        src = json.load(open(f'{FXL}/SOURCE.json'))
        assert src.get('parity'), 'parity fusion_cache_extra lỗi: xem log'
        publish(FXL, FCX, FCX_MAN, partitions=EXTRA, parity=src['parity'], kmedoids_train=km_x)
    FXL = fetch(FCX_MAN, f'/content/fusion_cache_x_{H8}')
    import torch
    for n in EXTRA:
        c = torch.load(f'{FXL}/train_{n}.pt', weights_only=False)
        assert len(c['video_ids']) == 1002 and bool(c['eg_ok'].all()), f"train {n}: {len(c['video_ids'])} video, c_g {int(c['eg_ok'].sum())}"
    print('fusion_cache_extra', FXL, '| parity:', [(r['partition'], r.get('expected R@1'), r['R@1 (E)']) for r in json.load(open(FCX_MAN))['parity']])
    manifest_add('8b fusion_cache_extra', {'MANIFEST.json': FCX_MAN, 'partitions': EXTRA}, t0)
else:
    print('RUN_EXTRA_FUSION = RUN_EXP3_FUSION = False: bỏ qua mục 8b')

In [ ]:
# 8b-2: train Context Fusion (D3, seed 0) cho từng dòng còn lại; bảng cột (c) đầy đủ; Exp 3 có Context Fusion
t0 = time.time()
def row_of(name, part, runs):
    r = load(name, runs)
    bs = r['val394 bootstrap (c)-(b) by video']
    return {'cách chia': LABEL[part], 'lượt': name, '(b)': r['val394 (b)']['R@1'], '(c)': r['val394 (c)']['R@1'],
            '(c) - (b)': bs['diff'], 'CI (video)': ci(bs['ci_low'], bs['ci_high']), 'ci_low': bs['ci_low'], 'alpha': r['alpha'],
            'VCMR (b)': r['val394 (b)']['VCMR R@1 (tIoU.5)'], 'VCMR (c)': r['val394 (c)']['VCMR R@1 (tIoU.5)'], 'epoch': r.get('best_epoch')}

for n in EXTRA:
    r = train(f'X_{n}', n, ['--seed 0'], runs=RUNSX, cache=FXL)
    tol = 0.1 if n == 'event_single_pred' else 0.006              # vector trung bình lưu fp16 (build_cache.POOLED_TOL)
    assert abs(r['val394 (b)']['R@1'] - expected_x[n]) <= tol, f"X_{n}: cột (b) {r['val394 (b)']['R@1']:.2f} khác {expected_x[n]}"
files = []
if RUN_EXTRA_FUSION:
    exp1_cx = pd.DataFrame([row_of(f'X_{n}', n, RUNSX) for n in EXTRA_C])
    exp1_cx.to_csv(f'{RES}/exp1_c_extra.csv', index=False); files.append('exp1_c_extra.csv')
    print('=== Exp 1 cột (c), các dòng còn lại (1 seed), val 394 ==='); display(exp1_cx.drop(columns='ci_low').round(2))
if RUN_EXP3_FUSION:
    e3 = pd.DataFrame([row_of('E_gt', 'gt', RUNSH)] + [row_of(f'X_{n}', n, RUNSX) for n in EXTRA_N])
    e3.insert(1, 'tIoU', [tiou['gt']] + [tiou[n] for n in EXTRA_N])
    e3.to_csv(f'{RES}/exp3_fusion.csv', index=False); files.append('exp3_fusion.csv')
    print('=== Exp 3 có Context Fusion (val 394; mức 1.0 = E_gt) ==='); display(e3.drop(columns='ci_low').round(3))
    top = e3.iloc[0]['(c) - (b)']
    low = e3[e3['lượt'].isin(['X_gt_noise_05', 'X_gt_noise_03'])]
    robust = bool(((low['(c) - (b)'] > top) & (low['ci_low'] > 0)).any())
    e3_verdict = {'quy tắc': '(c)-(b) ở mức 0.5 hoặc 0.3 lớn hơn ở mức 1.0 và CI ở mức đó không chứa 0',
                  'bền hơn với ranh giới kém': robust, '(c)-(b) ở mức 1.0': round(float(top), 3),
                  **{f"(c)-(b) {r['lượt']}": round(float(r['(c) - (b)']), 3) for _, r in low.iterrows()}}
    print(json.dumps(e3_verdict, ensure_ascii=False, indent=1))
    fig, ax = plt.subplots(figsize=(7, 4.5))
    d = e3.sort_values('tIoU')
    ax.plot(d['tIoU'], d['(b)'], 'o-', color='C0', label='không có Context Fusion (b)')
    ax.plot(d['tIoU'], d['(c)'], 's--', color='C3', label='có Context Fusion (c), 1 seed')
    ax.set_xlabel('tIoU trung bình với GT'); ax.set_ylabel('R@1 (%), gallery val 394'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    fig.savefig(f'{RES}/p5_fusion.png', dpi=150, bbox_inches='tight'); plt.show(); files.append('p5_fusion.png')
if EXTRA:
    manifest_add('8b extra fusion', {f: f'{RES}/{f}' for f in files}, t0, runs=RUNSX)

## 9. Exp 5: prior `conf`, cách gộp điểm, hiệu năng, int8 (CPU)

β chọn trên dev (hoà lấy β nhỏ hơn), val chấm một lần; `assert`: β = 0 trùng cột (b). `'old'`: β = 0.01, val 25.12, int8 R6 24.85.

In [ ]:
t0 = time.time()
from omni_retrieval.conf_scoring import conf_rows, index_for, pick_beta, sweep
from omni_retrieval.efficiency import OMNI_SEC_PER_CLIP, efficiency_row, int8_index
S = PROTO['scoring']['exp5']
rows5 = conf_rows(DB, f'{UG}/segments_pred.jsonl')
dev5 = pd.DataFrame(sweep(rows5, text_dev, S['beta_grid'], DEV))
BETA = pick_beta(dev5.to_dict('records'))
out5 = []
for gal, videos in (('val394', VAL), ('all498', None)):
    i0, ib = index_for(rows5, 0.0, videos), index_for(rows5, BETA, videos)
    k0, kb = video_ranks(i0, text)['ranks'], video_ranks(ib, text)['ranks']
    out5.append(dict(gallery=gal, beta=0.0, **evaluate_partition(i0, text)))
    bs = paired_bootstrap(kb, k0, 1, 1000, 0, 'video', text['video_id'])
    out5.append(dict(gallery=gal, beta=BETA, **evaluate_partition(ib, text), **{'Δ R@1': bs['diff'], 'CI': ci(bs['ci_low'], bs['ci_high'])}))
val5 = pd.DataFrame(out5)
dev5.to_csv(f'{RES}/exp5_dev.csv', index=False); val5.to_csv(f'{RES}/exp5_val.csv', index=False)
json.dump({'beta': BETA, 'rule': S['beta_rule']}, open(f'{RES}/exp5_beta.json', 'w'))
display(dev5[['beta', 'R@1', 'R@5', 'VCMR R@1 (tIoU.5)']].round(2)); print('β chọn trên dev:', BETA)
display(val5[['gallery', 'beta', 'R@1', 'R@5', 'VCMR R@1 (tIoU.5)', 'Δ R@1', 'CI']].round(2))
v0 = val5[(val5.gallery == 'val394') & (val5.beta == 0)]['R@1'].iloc[0]
assert abs(v0 - expected['uniav_pred']) < 0.006, 'β = 0 khác cột (b)'

# cách gộp điểm (R2, R3, R6, R7): dev, val 394
agg = []
for split, tx, videos in (('dev', text_dev, DEV), ('val394', text, VAL)):
    for p in ('uni_chunk', 'uni_M', 'uniav_pred', 'gt'):
        idx = load_index(DB, p, videos)
        base_r = video_ranks(idx, tx)['ranks']
        for name, kw in S['aggs']:
            r = evaluate_partition(idx, tx, name, **kw)
            row = {'split': split, 'partition': LABEL[p], 'agg': name + ''.join(f' {k}={v}' for k, v in kw.items()), 'R@1': r['R@1']}
            if name != 'max':
                bs = paired_bootstrap(video_ranks(idx, tx, name, **kw)['ranks'], base_r, 1, 1000, 0, 'video', tx['video_id'])
                row.update({'Δ vs max': bs['diff'], 'CI': ci(bs['ci_low'], bs['ci_high'])})
            agg.append(row)
agg = pd.DataFrame(agg)
agg.to_csv(f'{RES}/exp5_agg.csv', index=False)
display(agg.pivot_table(index='agg', columns=['split', 'partition'], values='R@1').round(2))

# hiệu năng (index 498 video) và int8 (val 394)
UNIAV_SEC = TIMING.get('detect s/video') if TIMING.get('videos') else None
q = text['emb'][:1000]
eff = pd.DataFrame([{**efficiency_row(load_index(DB, p), q, p in PROTO['partitions']['uses_uniav'], UNIAV_SEC), 'partition': LABEL[p]}
                    for p in ('global', 'event_single', 'event_single_pred', 'uni_chunk', 'uni_M', 'uniav_pred', 'gt', 'uniav_pred+uni_chunk')])
eff.to_csv(f'{RES}/exp5_efficiency.csv', index=False)
display(eff[[c for c in ('partition', 'vectors', 'fp16 (MB)', 'int8 (MB)', 'CPU 1 query (ms)', 'GPU 1 query (ms)',
                         'Omni s/video', 'UniAV s/video') if c in eff]].round(3))
i8 = []
for p in ('uniav_pred', 'uni_chunk', 'gt', 'uniav_pred+uni_chunk'):
    idx = load_index(DB, p, VAL)
    r16, r8 = video_ranks(idx, text)['ranks'], video_ranks(int8_index(idx), text)['ranks']
    bs = paired_bootstrap(r8, r16, 1, 1000, 0, 'video', text['video_id'])
    i8.append({'partition': p, 'R@1 fp16': 100 * float((r16 <= 1).mean()), 'R@1 int8': 100 * float((r8 <= 1).mean()),
               'Δ': bs['diff'], 'CI': ci(bs['ci_low'], bs['ci_high'])})
i8 = pd.DataFrame(i8)
i8.to_csv(f'{RES}/exp5_int8.csv', index=False); display(i8.round(2))
if OLD:
    vb = val5[(val5.gallery == 'val394') & (val5.beta == BETA)]['R@1'].iloc[0]
    check_ref({'beta': BETA, 'val': vb, 'int8': i8.set_index('partition').loc['uniav_pred', 'R@1 int8']},
              {'beta': REF['exp5_beta'], 'val': REF['exp5_val394_R@1_beta'], 'int8': REF['int8_val394_R@1_uniav_pred']}, 'Exp 5')
manifest_add('9 exp5', {f: f'{RES}/{f}' for f in ('exp5_dev.csv', 'exp5_val.csv', 'exp5_agg.csv', 'exp5_efficiency.csv', 'exp5_int8.csv')}, t0)

## 10. Phân tích định tính: A4, lặp bước trong cùng video, attention của bước 1

`qualitative.run` (bảng nhóm lật, ví dụ, near-duplicate, **đúng video sai event**, timeline) và trọng số cross-attention của D3 (seed 0) cho 3 video val. `assert`: R@1 tính lại trùng cột (b).

In [ ]:
t0 = time.time()
from omni_retrieval.qualitative import run as qual_run
QD = f'{RES}/qualitative'
out10 = qual_run(DB, VAL, PG, SUBSET, DUP_CSV, QD, n_examples=10, n_timelines=5)
dfq = out10['per_query']
r1q = {s: 100 * float((dfq[f'rank {s}'] == 1).mean()) for s in ('R0', 'R2', 'R3', 'R6', 'R7')}
check_ref(r1q, {s: r1['val394'][p] for s, p in (('R0', 'global'), ('R2', 'uni_chunk'), ('R3', 'uni_M'), ('R6', 'uniav_pred'), ('R7', 'gt'))},
          'A4: R@1 tính lại = cột (b)')
print('=== đặc điểm theo nhóm ==='); display(out10['profile'].round(2))
sv = out10['same_video_wrong_event']
print(f'=== đúng video, sai event: {len(sv)} query; jaccard >= 0.5 (nghi lặp bước): {int((sv.jaccard >= 0.5).sum()) if len(sv) else 0} ===')
display(sv.head(15))
for f in out10['figures'][:3]:
    display(Image(f))

# attention bước 1 của D3 (seed 0): 3 video val có nhiều đoạn nhất
from uemr_fusion.data import FusionSplit
from uemr_fusion.evaluate import attention_maps
from uemr_fusion.model import ContextFusion, FusionConfig
ck = torch.load(f'{RUNSH}/D3/best.pt', weights_only=False, map_location='cpu')
model = ContextFusion(FusionConfig(**ck['config']))
model.load_state_dict(ck['model'])
vs = FusionSplit(FCL, 'val', 'uniav_pred')
pick = sorted(range(vs.n_videos), key=lambda i: -int(vs.kmask[i].sum()))[:3]
maps = attention_maps(model, vs, pick)
fig, axes = plt.subplots(len(maps), 1, figsize=(13, 3 * len(maps)))
for ax, mp in zip(np.atleast_1d(axes), maps):
    keep = mp['t_sec'] <= dur[mp['video_id']] + 1
    ax.imshow(mp['attn'][:, keep], aspect='auto', cmap='magma', extent=(mp['t_sec'][keep][0], mp['t_sec'][keep][-1], len(mp['segs']) - 0.5, -0.5))
    for k, (a_, b_) in enumerate(mp['segs']):
        ax.plot([a_, b_], [k, k], color='cyan', lw=2)
    for ts, te in gt0[mp['video_id']]:
        ax.axvspan(ts, te, color='lime', alpha=0.08)
    ax.set_title(f"{mp['video_id']}: attention bước 1 (trung bình 8 head) của từng đoạn R6 (vạch xanh: đoạn; nền xanh lá: GT)")
    ax.set_xlabel('giây'); ax.set_ylabel('đoạn')
fig.tight_layout(); fig.savefig(f'{QD}/attention_step1.png', dpi=120); plt.show()
inside = [float(np.mean([mp['attn'][k][(mp['t_sec'] >= a_) & (mp['t_sec'] <= b_)].sum() for k, (a_, b_) in enumerate(mp['segs'])])) for mp in maps]
print('khối lượng attention rơi vào trong đoạn (trung bình mỗi video):', [round(x, 3) for x in inside])
manifest_add('10 qualitative', {'qual_profile.csv': f'{QD}/qual_profile.csv', 'same_video': f'{QD}/qual_same_video_wrong_event.csv',
                                'attention': f'{QD}/attention_step1.png'}, t0)

## 11. (Tuỳ chọn) Exp 4: audio phía Omni

Omni-fix có đầu vào chỉ-video (`_prepare_submodal_input({'video'})`) nhưng chỉ dùng cho Tuple-InfoNCE lúc train; bộ encode `av` và `checkpoint-1948` (InfoNCE cặp video + audio) không có nhánh đó. Ở đây audio của mọi clip được thay bằng **8 giây im lặng** (`encode --silence`, đúng nhánh dự phòng của loader): **đầu vào ngoài phân phối** của checkpoint, nên hiệu AV − im lặng là cận trên của đóng góp audio chứ không phải số đo "visual-only" sạch. R6 và R7, gallery val 394, store riêng `raw/av_silence`. Phía UniAV (Z chỉ hình) chưa làm.

In [ ]:
if RUN_OPTIONAL:
    t0 = time.time()
    enc = omni_env('ft')
    store = f'{O}/ft/raw/av_silence'
    rows11 = [r for r in prep_ft['rows'] if r['partition'] in ('gt', 'uniav_pred') and r['video_id'] in VAL]
    segs11 = {r['seg_key']: prep_ft['segments'][r['seg_key']] for r in rows11}
    todo11 = segment_records(segs11, store_keys(store, 'av'))
    print('clip im lặng cần encode:', len(todo11))
    if todo11:
        write_jsonl('/content/silence_todo.jsonl', todo11)
        copy_videos(sorted(VAL))
        encode('av', '/content/silence_todo.jsonl', store, f'{RES}/exp4_encode.log', enc, '--num-workers 4 --silence')
    from omni_retrieval.collect import collect_segments
    os.makedirs('/content/db_silence', exist_ok=True)
    print(collect_segments(rows11, store, Path('/content/db_silence/segments.npz')))
    shutil.copy2(f'{DB}/text.npz', '/content/db_silence/text.npz')
    e4 = []
    for p in ('uniav_pred', 'gt'):
        r_av, r_sil = RK[p], video_ranks(load_index('/content/db_silence', p, VAL), text)['ranks']
        bs = paired_bootstrap(r_av, r_sil, 1, 1000, 0, 'video', text['video_id'])
        e4.append({'partition': LABEL[p], 'R@1 audio thật': 100 * float((r_av <= 1).mean()), 'R@1 audio im lặng (OOD)': 100 * float((r_sil <= 1).mean()),
                   'AV - im lặng': bs['diff'], 'CI (video)': ci(bs['ci_low'], bs['ci_high'])})
    e4 = pd.DataFrame(e4)
    e4.to_csv(f'{RES}/exp4_silence.csv', index=False); display(e4.round(2))
    manifest_add('11 exp4', {'exp4_silence.csv': f'{RES}/exp4_silence.csv', 'note': 'audio = silence, out of distribution'}, t0)
else:
    print('RUN_OPTIONAL = False: bỏ qua Exp 4')

## 12. Tổng hợp: `results/<hash>/`, số liệu cho `UEMR_TONG_KET.md`, checklist §14

In [ ]:
t0 = time.time()
def md(df, cols=None, nd=2):
    d = (df[cols] if cols else df).copy()
    head = '| ' + ' | '.join(map(str, d.columns)) + ' |\n|' + '---|' * len(d.columns) + '\n'
    fmt = lambda x: f'{x:.{nd}f}' if isinstance(x, (float, np.floating)) and x == x else ('' if x is None or x != x else str(x))
    return head + '\n'.join('| ' + ' | '.join(fmt(x) for x in r) + ' |' for r in d.itertuples(index=False))
bv = b[b.split == 'val394'].assign(partition=lambda x: x.partition.map(LABEL))
parts_md = [f'# Số liệu UEMR, UniAV {UNIAV_SOURCE} ({H8})\n', f'Sinh bởi uemr_final_AZ.ipynb, code {COMMIT}, protocol {PROTO_SHA}.\n',
            '## UniAV\n', md(uni.reset_index()), f'\nθ = {theta_info["theta"]} (chọn trên {theta_info["chosen_on"]}); bỏ sót R6 (tIoU < 0.3): {MISS}\n',
            '## Exp 1 cột (b), val 394\n', md(bv, ['partition', 'vec/video', 'R@1', 'R@5', 'R@10', 'MedR', 'VCMR R@1 (tIoU.5)', 'mom@.5|top1']),
            '\n## Bootstrap theo video\n', md(boot[boot.split == 'val394'].replace(LABEL), ['A', 'B', 'A - B', 'CI (video)']),
            '\n## Cột (a) pt\n', md(a.assign(partition=lambda x: x.partition.map(LABEL)), ['partition', 'R@1', 'R@5', 'R@10']),
            '\n## Hồi quy độ phủ\n', md(reg), '\n## Exp 3\n', md(lev.drop(columns='ci_low R@1'), nd=3),
            f'\n{json.dumps(summary)}\n', '\n## Exp 1 cột (c)\n', md(exp1_c), f'\n{json.dumps(CF_VERDICT, ensure_ascii=False)}\n', '\n## Exp 2\n', md(exp2), '\n', md(pairs),
            '\n## Exp 5\n', md(val5, ['gallery', 'beta', 'R@1', 'R@5', 'Δ R@1', 'CI']), '\n', md(i8),
            '\n## A4\n', md(out10['profile'])]
if RUN_EXTRA_FUSION:
    parts_md += ['\n## Exp 1 cột (c), các dòng còn lại (1 seed)\n', md(exp1_cx)]
if RUN_EXP3_FUSION:
    parts_md += ['\n## Exp 3 có Context Fusion (1 seed mỗi mức)\n', md(e3), f'\n{json.dumps(e3_verdict, ensure_ascii=False)}\n']
open(f'{RES}/so_lieu.md', 'w').write('\n'.join(parts_md))
print(f'{RES}/so_lieu.md:', os.path.getsize(f'{RES}/so_lieu.md'), 'byte')
CHECK.update({'R3, R4, R5 dùng K_pred từng video': True, 'cùng level Z, d_min, cách cắt clip': True,
              'seed, config, hash ghi lại (run_manifest.json)': True,
              'Omni chỉ fine-tune trên train, chọn trên dev': 'có: train trên trainsplit; checkpoint-1948 cao nhất trên dev trong 9 checkpoint (omni_ckpt_select/decision.json)',
              'UniAV chỉ train trên train, chọn trên dev': 'không (Athena hiện tại thấy dev, chọn epoch trên val)' if OLD
                                                          else f'có: Athena sạch {ATHENA_H8} (train 1.002 trainsplit, best_seg / best_cap chọn trên dev; athena_sach/{ATHENA_H8}/SOURCE.json)'})
display(pd.DataFrame([{'§14': k, 'trạng thái': v} for k, v in CHECK.items()]))
manifest_add('12 summary', {'so_lieu.md': f'{RES}/so_lieu.md', 'checklist': CHECK, 'env': ENV}, t0)

## 13. So sánh UniAV cũ và mới (khi đã chạy cả hai)

Đọc `results/<hash cũ>/` và `results/<hash mới>/`; với mỗi dòng phụ thuộc UniAV: R@1 cũ, mới, hiệu và CI theo video (cùng query); cùng bảng cho F1, K_pred, tỉ lệ bỏ sót, cột (c), Exp 5.

In [ ]:
st = load_state()
if not ('old' in st and 'new' in st):
    print('cần chạy notebook với cả UNIAV_SOURCE = "old" và "new" trước')
else:
    R_OLD, R_NEW = f"{BASE}/results/{st['old']['hash8']}", RES   # mới: results/<ATHENA_H8>
    ko, kn = np.load(f'{R_OLD}/ranks_val394.npz'), np.load(f'{R_NEW}/ranks_val394.npz')
    assert np.array_equal(ko['caption_id'], kn['caption_id'])
    cmp = []
    for p in ('uni_M', 'rand_M', 'kmedoids_M', 'uniav_pred', 'uniav_shift', 'uniav_pred+uni_chunk', 'event_single_pred'):
        bs = paired_bootstrap(kn[p], ko[p], 1, 1000, 0, 'video', ko['video_id'])
        cmp.append({'mục': 'Exp 1 (b) R@1', 'dòng': LABEL[p], 'cũ': 100 * float((ko[p] <= 1).mean()),
                    'mới': 100 * float((kn[p] <= 1).mean()), 'mới - cũ': bs['diff'], 'CI (video)': ci(bs['ci_low'], bs['ci_high'])})
    uo, un = pd.read_csv(f'{R_OLD}/uniav_stats.csv', index_col=0), pd.read_csv(f'{R_NEW}/uniav_stats.csv', index_col=0)
    for s in ('trainsplit', 'dev', 'val'):
        for col in ('F1@0.5', 'AP@0.5', 'K_pred mean', 'GT bỏ sót % (tIoU < 0.3)'):
            cmp.append({'mục': f'UniAV {col}', 'dòng': s, 'cũ': uo.loc[s, col], 'mới': un.loc[s, col], 'mới - cũ': un.loc[s, col] - uo.loc[s, col]})
    for f, key in (('exp1_c.csv', '(c) - (b)'),):
        co, cn = pd.read_csv(f'{R_OLD}/{f}').set_index('lượt'), pd.read_csv(f'{R_NEW}/{f}').set_index('lượt')
        for n in co.index.intersection(cn.index):
            cmp.append({'mục': 'Exp 1 (c) − (b)', 'dòng': n, 'cũ': co.loc[n, key], 'mới': cn.loc[n, key],
                        'CI (video)': f"cũ {co.loc[n, 'CI mean seed (video)']} | mới {cn.loc[n, 'CI mean seed (video)']}"})
    cmp = pd.DataFrame(cmp)
    cmp.to_csv(f'{R_NEW}/old_vs_new.csv', index=False)
    display(cmp.round(2))

    # bảng cũ / mới cho UEMR_KET_QUA.md (docs/UEMR_B_CHAY_LAI_AZ.md việc 4)
    bo, bn = (pd.read_csv(f'{R}/exp1_b.csv') for R in (R_OLD, R_NEW))
    so, sn = (pd.read_csv(f'{R}/exp1_b_bootstrap.csv') for R in (R_OLD, R_NEW))
    co_, cn_ = (pd.read_csv(f'{R}/exp1_c.csv').set_index('lượt') for R in (R_OLD, R_NEW))
    def rr(df, p, col='R@1', split='val394'):
        return float(df[(df.split == split) & (df.partition == p)][col].iloc[0])
    def bsr(df, A, B):
        return df[(df.split == 'val394') & (df.A == A) & (df.B == B)].iloc[0]
    both = (('cũ ' + st['old']['hash8'], bo, so, co_), ('mới ' + ATHENA_H8, bn, sn, cn_))
    tab = [{'Dòng': 'R0 / R3 / R2 / R7 R@1', **{k: ' / '.join(f'{rr(b_, p):.2f}' for p in ('global', 'uni_M', 'uni_chunk', 'gt')) for k, b_, s_, c_ in both}},
           {'Dòng': 'R6 R@1 / R@5 / R@10', **{k: ' / '.join(f'{rr(b_, "uniav_pred", c):.2f}' for c in ('R@1', 'R@5', 'R@10')) for k, b_, s_, c_ in both}},
           {'Dòng': 'R6 VCMR R@1', **{k: f'{rr(b_, "uniav_pred", "VCMR R@1 (tIoU.5)"):.2f}' for k, b_, s_, c_ in both}},
           *({'Dòng': f'R6 − {n}, CI 95%', **{k: f"{bsr(s_, 'uniav_pred', p)['A - B']:+.2f} {bsr(s_, 'uniav_pred', p)['CI (video)']}" for k, b_, s_, c_ in both}}
             for n, p in (('R3', 'uni_M'), ('R0', 'global'))),
           {'Dòng': 'R6 ∪ R2 R@1', **{k: f'{rr(b_, "uniav_pred+uni_chunk"):.2f}' for k, b_, s_, c_ in both}},
           {'Dòng': 'R6 + CF R@1 (3 seed); Δ so với R6, CI 95%',
            **{k: f"{c_.loc['D3', '(c) mean']:.2f}; {c_.loc['D3', '(c) - (b)']:+.2f} {c_.loc['D3', 'CI mean seed (video)']}" for k, b_, s_, c_ in both}},
           {'Dòng': 'R6 R@1, kho 498', **{k: f'{rr(b_, "uniav_pred", split="all498"):.2f}' for k, b_, s_, c_ in both}}]
    tab = pd.DataFrame(tab)
    tab.to_csv(f'{R_NEW}/bang_cu_moi.csv', index=False)
    display(tab)
    ORDER = {f'R6 > {n} (cận dưới CI > 0)': bool(bsr(sn, 'uniav_pred', p)['ci_low'] > 0) for n, p in (('R3', 'uni_M'), ('R0', 'global'))}
    print('thứ tự với Athena sạch:', ORDER, '| Context Fusion:', CF_VERDICT['phương pháp chính'])
    json.dump({'order': ORDER, 'cf': CF_VERDICT}, open(f'{R_NEW}/ket_luan.json', 'w'), indent=1, ensure_ascii=False)


## In lại mọi bảng để gửi

In [ ]:
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
print(open(f'{RES}/so_lieu.md').read() if os.path.exists(f'{RES}/so_lieu.md') else '(chưa chạy mục 12)')
print(json.dumps(json.load(open(f'{RES}/run_manifest.json'))['steps'][-12:], indent=1, ensure_ascii=False)[:6000])